# Агент генерации клиентской презентации

Прототип строит презентацию по схеме:

1. загрузка локальных данных;
2. создание эмбеддингов через GigaChat;
3. индексация документов в OpenSearch;
4. гибридный RAG-поиск по клиенту, продуктам и новостям;
5. генерация строго структурированного плана;
6. программная сборка `.pptx`.

Проект предполагается расположенным в `D:/sber`, а исходные файлы — в `D:/sber/data`.


## 1. Установка зависимостей

In [5]:
%pip install -q pandas python-dotenv opensearch-py gigachat \
    python-pptx pydantic tenacity sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Импорты и конфигурация

In [31]:
from __future__ import annotations

import json
import os
import re
import time
import hashlib
import warnings
from dataclasses import dataclass
from difflib import get_close_matches
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from dotenv import load_dotenv
from opensearchpy import OpenSearch, helpers
from pydantic import BaseModel, Field, ValidationError
from tenacity import retry, stop_after_attempt, wait_exponential

from gigachat import GigaChat

from pptx import Presentation
from pptx.dml.color import RGBColor
from pptx.enum.shapes import MSO_SHAPE
from pptx.enum.text import MSO_AUTO_SIZE, MSO_ANCHOR, PP_ALIGN
from pptx.util import Inches, Pt

warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_DIR = Path(r"D:/sber")
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_DIR / ".env")

GIGACHAT_CREDENTIALS = os.getenv("GIGACHAT_CREDENTIALS", "").strip()
GIGACHAT_SCOPE = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_PERS")
GIGACHAT_BASE_URL = os.getenv("GIGACHAT_BASE_URL", "https://api.giga.chat/v1")
GIGACHAT_CHAT_MODEL = os.getenv("GIGACHAT_CHAT_MODEL", "GigaChat-2-Max")

GIGACHAT_VERIFY_SSL = (
    os.getenv("GIGACHAT_VERIFY_SSL", "false").lower() == "true"
)

OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST", "localhost")
OPENSEARCH_PORT = int(os.getenv("OPENSEARCH_PORT", "9200"))
OPENSEARCH_INDEX = os.getenv(
    "OPENSEARCH_INDEX",
    "client-presentation-rag",
)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_DIR: D:\sber
DATA_DIR: D:\sber\data
OUTPUT_DIR: D:\sber\outputs


## 3. Проверка файлов и загрузка данных

In [3]:
REQUIRED_FILES = {
    "clients": DATA_DIR / "clients.json",
    "products": DATA_DIR / "products.json",
    "internal_notes": DATA_DIR / "internal_notes.json",
    "news": DATA_DIR / "news.csv",
}


def validate_project_files() -> None:
    missing = [
        str(path)
        for path in REQUIRED_FILES.values()
        if not path.exists()
    ]
    if missing:
        raise FileNotFoundError(
            "Не найдены обязательные файлы:\n"
            + "\n".join(missing)
        )

    if not GIGACHAT_CREDENTIALS:
        raise RuntimeError(
            "В D:/sber/.env не задан GIGACHAT_CREDENTIALS."
        )


def load_json(path: Path) -> Any:
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_source_data() -> dict[str, Any]:
    validate_project_files()

    clients = load_json(REQUIRED_FILES["clients"])
    products = load_json(REQUIRED_FILES["products"])
    internal_notes = load_json(REQUIRED_FILES["internal_notes"])
    news = pd.read_csv(REQUIRED_FILES["news"])

    required_news_columns = {"title", "text", "source"}
    missing_columns = required_news_columns - set(news.columns)
    if missing_columns:
        raise ValueError(
            "В news.csv отсутствуют колонки: "
            + ", ".join(sorted(missing_columns))
        )

    news = news.fillna("")
    for column in required_news_columns:
        news[column] = news[column].astype(str).str.strip()

    return {
        "clients": clients,
        "products": products,
        "internal_notes": internal_notes,
        "news": news,
    }


source_data = load_source_data()

print("Клиентов:", len(source_data["clients"]))
print("Продуктов:", len(source_data["products"]))
print("Внутренних заметок:", len(source_data["internal_notes"]))
print("Новостей:", len(source_data["news"]))
print("Клиенты:", ", ".join(source_data["clients"].keys()))


Клиентов: 8
Продуктов: 10
Внутренних заметок: 8
Новостей: 1880
Клиенты: РусГидро, Аэрофлот, Норникель, РЖД, Магнит, Лукойл, МТС, ФосАгро


## 4. Нормализация документов

Каждая запись превращается в документ единого формата. Для новостей дополнительно
определяется, к каким клиентам она относится. На этапе поиска новостей используется
обязательный фильтр по `client_names`.


In [4]:
CLIENT_ALIASES = {
    "РусГидро": ["русгидро", "пао русгидро"],
    "Аэрофлот": ["аэрофлот", "группа аэрофлот"],
    "Норникель": [
        "норникель",
        "норильский никель",
        "гмк норильский никель",
    ],
    "РЖД": [
        "ржд",
        "российские железные дороги",
        "оао российские железные дороги",
    ],
    "Магнит": ["магнит", "пао магнит"],
    "Лукойл": ["лукойл", "пао лукойл"],
    "МТС": [
        "мтс",
        "мобильные телесистемы",
        "пао мтс",
    ],
    "ФосАгро": ["фосагро", "пао фосагро"],
}


def normalize_text(text: str) -> str:
    text = str(text).lower().replace("ё", "е")
    text = re.sub(r"[^a-zа-я0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def stable_id(*parts: str) -> str:
    raw = "||".join(str(part) for part in parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()


def detect_clients_in_text(
    text: str,
    client_names: Iterable[str],
) -> list[str]:
    normalized = f" {normalize_text(text)} "
    found = []

    for client_name in client_names:
        aliases = CLIENT_ALIASES.get(
            client_name,
            [client_name.lower()],
        )
        aliases = list(aliases) + [client_name.lower()]

        if any(
            f" {normalize_text(alias)} " in normalized
            for alias in aliases
        ):
            found.append(client_name)

    return sorted(set(found))


def detect_note_client(
    note: str,
    client_names: Iterable[str],
) -> list[str]:
    return detect_clients_in_text(note, client_names)


def build_documents(data: dict[str, Any]) -> list[dict[str, Any]]:
    clients = data["clients"]
    products = data["products"]
    internal_notes = data["internal_notes"]
    news = data["news"]

    documents: list[dict[str, Any]] = []

    for client_name, description in clients.items():
        documents.append({
            "doc_id": stable_id("client", client_name),
            "doc_type": "client",
            "title": client_name,
            "text": description,
            "source": "clients.json",
            "client_names": [client_name],
        })

    for product_name, description in products.items():
        documents.append({
            "doc_id": stable_id("product", product_name),
            "doc_type": "product",
            "title": product_name,
            "text": description,
            "source": "products.json",
            "client_names": [],
        })

    for index, note in enumerate(internal_notes):
        note_clients = detect_note_client(
            note,
            clients.keys(),
        )
        documents.append({
            "doc_id": stable_id("internal_note", str(index), note),
            "doc_type": "internal_note",
            "title": (
                f"Внутренняя заметка: {', '.join(note_clients)}"
                if note_clients
                else "Внутренняя заметка"
            ),
            "text": note,
            "source": "internal_notes.json",
            "client_names": note_clients,
        })

    for index, row in news.iterrows():
        combined_text = f"{row['title']}\n{row['text']}"
        news_clients = detect_clients_in_text(
            combined_text,
            clients.keys(),
        )
        documents.append({
            "doc_id": stable_id(
                "news",
                str(index),
                row["title"],
                row["source"],
            ),
            "doc_type": "news",
            "title": row["title"],
            "text": row["text"],
            "source": row["source"],
            "client_names": news_clients,
        })

    return documents


documents = build_documents(source_data)

documents_df = pd.DataFrame(documents)
display(
    documents_df[
        ["doc_type", "title", "source", "client_names"]
    ].head(20)
)

print("\nРаспределение документов:")
print(documents_df["doc_type"].value_counts())

unassigned_news = documents_df[
    (documents_df["doc_type"] == "news")
    & (documents_df["client_names"].map(len) == 0)
]
print("\nНовостей без найденного клиента:", len(unassigned_news))


,doc_type,title,source,client_names
0,client,РусГидро,clients.json,[РусГидро]
1,client,Аэрофлот,clients.json,[Аэрофлот]
2,client,Норникель,clients.json,[Норникель]
3,client,РЖД,clients.json,[РЖД]
4,client,Магнит,clients.json,[Магнит]
5,client,Лукойл,clients.json,[Лукойл]
6,client,МТС,clients.json,[МТС]
7,client,ФосАгро,clients.json,[ФосАгро]
8,product,IRS,products.json,[]
9,product,XCCY,products.json,[]



Распределение документов:
doc_type
news             1880
product            10
client              8
internal_note       8
Name: count, dtype: int64

Новостей без найденного клиента: 1841


## 5. Обертка над GigaChat

In [5]:
def extract_chat_text(response: Any) -> str:
    if hasattr(response, "choices") and response.choices:
        message = response.choices[0].message
        content = getattr(message, "content", None)
        if isinstance(content, str):
            return content

    if hasattr(response, "messages") and response.messages:
        parts = response.messages[0].content
        if parts:
            text = getattr(parts[0], "text", None)
            if text:
                return text

    if isinstance(response, dict):
        choices = response.get("choices") or []
        if choices:
            return choices[0]["message"]["content"]

    raise ValueError(
        "Не удалось извлечь текст из ответа GigaChat."
    )


def extract_embeddings(response: Any) -> list[list[float]]:
    data = getattr(response, "data", None)

    if data is None and isinstance(response, dict):
        data = response.get("data")

    if data is None:
        raise ValueError(
            "Не удалось извлечь embeddings из ответа GigaChat."
        )

    vectors = []
    for item in data:
        vector = (
            getattr(item, "embedding", None)
            if not isinstance(item, dict)
            else item.get("embedding")
        )
        if vector is None:
            raise ValueError("В ответе отсутствует поле embedding.")
        vectors.append(list(vector))

    return vectors


@dataclass
class GigaChatService:
    credentials: str
    scope: str = GIGACHAT_SCOPE
    base_url: str = GIGACHAT_BASE_URL
    chat_model: str = GIGACHAT_CHAT_MODEL
    verify_ssl_certs: bool = GIGACHAT_VERIFY_SSL

    def _client(self) -> GigaChat:
        return GigaChat(
            credentials=self.credentials,
            scope=self.scope,
            base_url=self.base_url,
            verify_ssl_certs=self.verify_ssl_certs,
        )

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(
            multiplier=1,
            min=2,
            max=10,
        ),
        reraise=True,
    )
    

    @retry(
        stop=stop_after_attempt(3),
        wait=wait_exponential(
            multiplier=1,
            min=2,
            max=10,
        ),
        reraise=True,
    )
    def generate(
        self,
        system_prompt: str,
        user_prompt: str,
        temperature: float = 0.1,
    ) -> str:
        from gigachat.models import (
            Chat,
            Messages,
            MessagesRole,
        )

        chat = Chat(
            model=self.chat_model,
            messages=[
                Messages(
                    role=MessagesRole.SYSTEM,
                    content=system_prompt,
                ),
                Messages(
                    role=MessagesRole.USER,
                    content=user_prompt,
                ),
            ],
            temperature=temperature,
        )

        with self._client() as client:
            response = client.chat(chat)

        return extract_chat_text(response)


giga = GigaChatService(
    credentials=GIGACHAT_CREDENTIALS,
)

# Небольшой тест соединения. Раскомментируйте при необходимости.
test_answer = giga.generate(
    system_prompt=(
        "Ты проверяешь работу API. "
        "Отвечай максимально кратко."
    ),
    user_prompt=(
        "Напиши только одно слово: работает"
    ),
    temperature=0.0,
)

print(test_answer)


работает


In [6]:
from sentence_transformers import SentenceTransformer


class LocalEmbeddingService:
    """
    Локальные эмбеддинги для документов и поисковых запросов.

    Модель загружается с Hugging Face при первом запуске,
    а затем сохраняется в локальном кэше.
    """

    def __init__(
        self,
        model_name: str = (
            "intfloat/multilingual-e5-small"
        ),
        device: str = "cpu",
    ) -> None:
        self.model_name = model_name

        self.model = SentenceTransformer(
            model_name,
            device=device,
        )

        self.dimension = (
            self.model
            .get_sentence_embedding_dimension()
        )

        if self.dimension is None:
            raise RuntimeError(
                "Не удалось определить размер эмбеддинга."
            )

    def embed_documents(
        self,
        texts: list[str],
        batch_size: int = 32,
    ) -> list[list[float]]:
        if not texts:
            return []

        prepared_texts = [
            f"passage: {text}"
            for text in texts
        ]

        vectors = self.model.encode(
            prepared_texts,
            batch_size=batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        return vectors.astype("float32").tolist()

    def embed_query(
        self,
        text: str,
    ) -> list[float]:
        prepared_text = f"query: {text}"

        vector = self.model.encode(
            prepared_text,
            normalize_embeddings=True,
            convert_to_numpy=True,
        )

        return vector.astype("float32").tolist()


embeddings_service = LocalEmbeddingService()

print("Локальная модель загружена.")
print("Модель:", embeddings_service.model_name)
print("Размерность:", embeddings_service.dimension)

C:\Users\Devadevam\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3795.68it/s]


Локальная модель загружена.
Модель: intfloat/multilingual-e5-small
Размерность: 384


C:\Users\Devadevam\AppData\Local\Temp\ipykernel_21048\3107452174.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  .get_sentence_embedding_dimension()


In [7]:
test_vectors = embeddings_service.embed_documents(
    [
        "РЖД имеет крупную инвестиционную программу.",
        "Аэрофлот чувствителен к валютному курсу.",
    ]
)

print("Количество векторов:", len(test_vectors))
print("Размер вектора:", len(test_vectors[0]))
print("Первые значения:", test_vectors[0][:5])

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.72it/s]

Количество векторов: 2
Размер вектора: 384
Первые значения: [0.0054777455516159534, 0.008398916572332382, -0.005311830434948206, -0.06066180765628815, 0.07049740850925446]


## 6. Подключение к OpenSearch

In [8]:
def create_opensearch_client() -> OpenSearch:
    client = OpenSearch(
        hosts=[{
            "host": OPENSEARCH_HOST,
            "port": OPENSEARCH_PORT,
        }],
        http_compress=True,
        use_ssl=False,
        verify_certs=False,
        ssl_assert_hostname=False,
        ssl_show_warn=False,
        timeout=60,
    )

    if not client.ping():
        raise ConnectionError(
            "OpenSearch недоступен. Запустите в D:/sber: "
            "`docker compose up -d`."
        )

    return client


opensearch = create_opensearch_client()
print(opensearch.info()["version"])


{'distribution': 'opensearch', 'number': '2.19.6', 'build_type': 'tar', 'build_hash': '97d3c13bf22a4a72ac11dc503fe44c97662b9161', 'build_date': '2026-07-01T04:27:16.038403104Z', 'build_snapshot': False, 'lucene_version': '9.12.3', 'minimum_wire_compatibility_version': '7.10.0', 'minimum_index_compatibility_version': '7.0.0'}


## 7. Создание индекса и загрузка документов

In [9]:
def embedding_text(document: dict[str, Any]) -> str:
    client_part = (
        ", ".join(document.get("client_names", []))
        or "не указан"
    )
    return (
        f"Тип: {document['doc_type']}\n"
        f"Клиент: {client_part}\n"
        f"Заголовок: {document['title']}\n"
        f"Текст: {document['text']}\n"
        f"Источник: {document['source']}"
    )


def create_index(
    client: OpenSearch,
    dimension: int,
    recreate: bool = False,
) -> None:
    exists = client.indices.exists(
        index=OPENSEARCH_INDEX,
    )

    if exists and recreate:
        client.indices.delete(
            index=OPENSEARCH_INDEX,
        )
        exists = False

    if exists:
        return

    index_body = {
        "settings": {
            "index": {
                "knn": True,
                "number_of_shards": 1,
                "number_of_replicas": 0,
            }
        },
        "mappings": {
            "properties": {
                "doc_id": {"type": "keyword"},
                "doc_type": {"type": "keyword"},
                "client_names": {"type": "keyword"},
                "title": {
                    "type": "text",
                    "analyzer": "russian",
                },
                "text": {
                    "type": "text",
                    "analyzer": "russian",
                },
                "source": {"type": "keyword"},
                "embedding": {
                    "type": "knn_vector",
                    "dimension": dimension,
                    "space_type": "cosinesimil",
                },
            }
        },
    }

    client.indices.create(
        index=OPENSEARCH_INDEX,
        body=index_body,
    )


def batch_iterable(
    values: list[Any],
    batch_size: int,
):
    for start in range(0, len(values), batch_size):
        yield values[start:start + batch_size]


def index_documents(
    client: OpenSearch,
    documents_to_index: list[dict[str, Any]],
    rebuild_index: bool = True,
    batch_size: int = 32,
) -> None:
    if not documents_to_index:
        raise ValueError(
            "Список документов пуст."
        )

    create_index(
        client=client,
        dimension=embeddings_service.dimension,
        recreate=rebuild_index,
    )

    indexed = 0

    for batch_number, batch in enumerate(
        batch_iterable(
            documents_to_index,
            batch_size,
        ),
        start=1,
    ):
        texts = [
            embedding_text(document)
            for document in batch
        ]

        vectors = (
            embeddings_service
            .embed_documents(
                texts,
                batch_size=batch_size,
            )
        )

        actions = []

        for document, vector in zip(
            batch,
            vectors,
        ):
            source = dict(document)
            source["embedding"] = vector

            actions.append({
                "_op_type": "index",
                "_index": OPENSEARCH_INDEX,
                "_id": document["doc_id"],
                "_source": source,
            })

        helpers.bulk(
            client,
            actions,
            refresh=False,
            request_timeout=120,
        )

        indexed += len(actions)

        print(
            f"Пакет {batch_number}: "
            f"проиндексировано "
            f"{indexed}/"
            f"{len(documents_to_index)}"
        )

    client.indices.refresh(
        index=OPENSEARCH_INDEX,
    )

    print("Индексация завершена.")


# Первый запуск:
# index_documents(opensearch, documents, rebuild_index=True)


## 8. Гибридный поиск

Выполняются два независимых запроса:

- BM25 по `title` и `text`;
- k-NN по эмбеддингам.

Результаты объединяются через Reciprocal Rank Fusion.


In [10]:
def build_filters(
    doc_type: str | None = None,
    client_name: str | None = None,
) -> list[dict[str, Any]]:
    filters = []

    if doc_type:
        filters.append({
            "term": {"doc_type": doc_type}
        })

    if client_name:
        filters.append({
            "term": {"client_names": client_name}
        })

    return filters


def lexical_search(
    client: OpenSearch,
    query_text: str,
    doc_type: str | None = None,
    client_name: str | None = None,
    size: int = 20,
) -> list[dict[str, Any]]:
    filters = build_filters(
        doc_type=doc_type,
        client_name=client_name,
    )

    body = {
        "size": size,
        "_source": {
            "excludes": ["embedding"]
        },
        "query": {
            "bool": {
                "must": [{
                    "multi_match": {
                        "query": query_text,
                        "fields": [
                            "title^4",
                            "text^2",
                        ],
                        "type": "best_fields",
                    }
                }],
                "filter": filters,
            }
        },
    }

    response = client.search(
        index=OPENSEARCH_INDEX,
        body=body,
    )
    return response["hits"]["hits"]


def vector_search(
    client: OpenSearch,
    query_text: str,
    doc_type: str | None = None,
    client_name: str | None = None,
    size: int = 50,
) -> list[dict[str, Any]]:
    query_vector = (
    embeddings_service
    .embed_query(query_text)
    )

    body = {
        "size": size,
        "_source": {
            "excludes": ["embedding"]
        },
        "query": {
            "knn": {
                "embedding": {
                    "vector": query_vector,
                    "k": size,
                }
            }
        },
    }

    response = client.search(
        index=OPENSEARCH_INDEX,
        body=body,
    )
    hits = response["hits"]["hits"]

    # Универсальная постфильтрация работает и для версий
    # OpenSearch, где фильтр внутри k-NN зависит от engine.
    filtered = []
    for hit in hits:
        source = hit["_source"]

        if doc_type and source["doc_type"] != doc_type:
            continue

        if (
            client_name
            and client_name
            not in source.get("client_names", [])
        ):
            continue

        filtered.append(hit)

    return filtered


def reciprocal_rank_fusion(
    ranked_lists: list[list[dict[str, Any]]],
    top_k: int,
    rrf_k: int = 60,
) -> list[dict[str, Any]]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, dict[str, Any]] = {}

    for ranked_hits in ranked_lists:
        for rank, hit in enumerate(
            ranked_hits,
            start=1,
        ):
            doc_id = hit["_id"]
            scores[doc_id] = (
                scores.get(doc_id, 0.0)
                + 1.0 / (rrf_k + rank)
            )
            documents_by_id[doc_id] = hit["_source"]

    ordered_ids = sorted(
        scores,
        key=scores.get,
        reverse=True,
    )[:top_k]

    return [
        {
            **documents_by_id[doc_id],
            "rag_score": scores[doc_id],
        }
        for doc_id in ordered_ids
    ]


def hybrid_search(
    client: OpenSearch,
    query_text: str,
    doc_type: str | None = None,
    client_name: str | None = None,
    top_k: int = 5,
) -> list[dict[str, Any]]:
    lexical_hits = lexical_search(
        client=client,
        query_text=query_text,
        doc_type=doc_type,
        client_name=client_name,
        size=max(20, top_k * 4),
    )

    vector_hits = vector_search(
        client=client,
        query_text=query_text,
        doc_type=doc_type,
        client_name=client_name,
        size=max(50, top_k * 8),
    )

    return reciprocal_rank_fusion(
        [lexical_hits, vector_hits],
        top_k=top_k,
    )


# Пример после индексации:
# hybrid_search(
#     opensearch,
#     "РЖД стоимость фондирования инвестиционная программа",
#     doc_type="product",
#     top_k=5,
# )


## 9. Формирование RAG-контекста

In [21]:
def get_client_profile(
    data: dict[str, Any],
    client_name: str,
) -> str:
    clients = data["clients"]

    if client_name not in clients:
        close = get_close_matches(
            client_name,
            clients.keys(),
            n=3,
            cutoff=0.4,
        )
        hint = (
            f" Возможно, имелось в виду: {', '.join(close)}."
            if close
            else ""
        )
        raise KeyError(
            f"Клиент '{client_name}' не найден.{hint}"
        )

    return clients[client_name]


SENSITIVE_NEWS_PATTERNS = [
    r"\bфронт\w*",
    r"\bвоенн\w*",
    r"\bоруж\w*",
    r"\bвзрыв\w*",
    r"\bракет\w*",
    r"\bбпла\b",
    r"\bдрон\w*",
    r"\bбоеприпас\w*",
    r"\bогнев\w*",
    r"\bубийств\w*",
    r"\bтеррор\w*",
    r"\bмобилизац\w*",
    r"\bвойн\w*",
]


def contains_client_alias(
    text: str,
    client_name: str,
) -> bool:
    normalized_text = (
        " "
        + normalize_text(text)
        + " "
    )

    aliases = CLIENT_ALIASES.get(
        client_name,
        [client_name],
    )

    aliases = list(aliases) + [client_name]

    for alias in aliases:
        normalized_alias = normalize_text(alias)

        if (
            f" {normalized_alias} "
            in normalized_text
        ):
            return True

    return False


def is_sensitive_news(
    item: dict[str, Any],
) -> bool:
    combined_text = normalize_text(
        f"{item['title']} {item['text']}"
    )

    return any(
        re.search(
            pattern,
            combined_text,
            flags=re.IGNORECASE,
        )
        for pattern in SENSITIVE_NEWS_PATTERNS
    )


def filter_safe_direct_news(
    news_items: list[dict[str, Any]],
    client_name: str,
    max_items: int = 4,
) -> list[dict[str, Any]]:
    """
    Новость считается прямой, только если клиент
    упомянут непосредственно в заголовке.

    Это отсекает статьи, где РЖД упоминается
    одним предложением в середине текста.
    """
    result = []

    for item in news_items:
        if not contains_client_alias(
            item["title"],
            client_name,
        ):
            continue

        if is_sensitive_news(item):
            continue

        result.append(item)

        if len(result) >= max_items:
            break

    return result



def build_rag_context(
    client: OpenSearch,
    data: dict[str, Any],
    client_name: str,
) -> dict[str, Any]:
    profile = get_client_profile(
        data,
        client_name,
    )

    note_query = (
        f"{client_name}. Внутренние рекомендации, "
        "приоритеты, ограничения и продуктовые идеи."
    )
    notes = hybrid_search(
        client,
        note_query,
        doc_type="internal_note",
        client_name=client_name,
        top_k=2,
    )

    note_text = "\n".join(
        item["text"] for item in notes
    )

    product_query = (
        f"Клиент: {client_name}.\n"
        f"Профиль: {profile}\n"
        f"Внутренние рекомендации: {note_text}\n"
        "Нужно подобрать продукты для управления "
        "рисками, фондированием и ликвидностью."
    )
    products = hybrid_search(
        client,
        product_query,
        doc_type="product",
        top_k=7,
    )

    news_query = (
        f"{client_name}: новости, инвестиционная "
        "программа, долговая нагрузка, финансирование, "
        "валютные платежи и ликвидность"
    )
    news = hybrid_search(
        client,
        news_query,
        doc_type="news",
        client_name=client_name,
        top_k=6,
    )

    safe_news = filter_safe_direct_news(
    news_items=news,
    client_name=client_name,
    max_items=4,
    )

    return {
    "client_name": client_name,
    "profile": profile,
    "notes": notes,
    "products": products,
    "news": safe_news,
    }


def print_context(context: dict[str, Any]) -> None:
    print("КЛИЕНТ:", context["client_name"])
    print("\nПРОФИЛЬ:\n", context["profile"])

    print("\nЗАМЕТКИ:")
    for item in context["notes"]:
        print("-", item["text"])

    print("\nКАНДИДАТЫ-ПРОДУКТЫ:")
    for item in context["products"]:
        print("-", item["title"])

    print("\nНОВОСТИ:")
    for item in context["news"]:
        print(
            "-",
            item["title"],
            "|",
            item["source"],
        )


# Пример:
# context = build_rag_context(
#     opensearch,
#     source_data,
#     "РЖД",
# )
# print_context(context)


## 10. Схема результата GigaChat

In [12]:
class RiskItem(BaseModel):
    name: str = Field(
        description="Короткое название риска"
    )
    impact: str = Field(
        description="Почему риск важен для клиента"
    )


class ProductRecommendation(BaseModel):
    name: str = Field(
        description="Точное название продукта из контекста"
    )
    fit: str = Field(
        description="Основная тема, Дополнительная тема или Требует подтверждения"
    )
    rationale: str = Field(
        description="Привязка продукта к задаче клиента"
    )
    caveat: str = Field(
        description="Условие или ограничение применения"
    )


class NewsItem(BaseModel):
    title: str = Field(
        description="Точный заголовок новости из контекста"
    )
    source: str = Field(
        description="Точный источник из контекста"
    )
    relevance: str = Field(
        description="Связь новости с финансовыми задачами клиента"
    )


class PresentationPlan(BaseModel):
    client_name: str
    subtitle: str
    profile_summary: str
    financial_tasks: list[str]
    risks: list[RiskItem]
    products: list[ProductRecommendation]
    news: list[NewsItem]
    meeting_questions: list[str]
    next_steps: list[str]


## 11. Генерация и проверка плана презентации

In [24]:
SYSTEM_PROMPT = """
Ты — технический редактор внутренних корпоративных
презентаций.

Тебе передан уже подготовленный набор фактов,
кандидатов-продуктов и новостей.

Ты не принимаешь финансовых решений, не выбираешь
инвестиционную стратегию и не даешь рекомендаций.
Твоя задача — только:

- сократить переданный текст;
- структурировать его для презентации;
- сделать формулировки нейтральными;
- сохранить ограничения из исходных данных.

Продукты называй:
- кандидатами для обсуждения;
- возможными темами встречи;
- предварительными продуктовыми идеями.

Не называй их инвестиционными рекомендациями.

Используй только предоставленные данные.
Не добавляй внешние сведения.
Не придумывай показатели, новости или сделки.

Верни только один валидный JSON-объект.
Первый символ ответа должен быть {
Последний символ ответа должен быть }
Не используй Markdown.
""".strip()


def format_documents(
    documents_list: list[dict[str, Any]],
) -> str:
    if not documents_list:
        return "Нет найденных документов."

    blocks = []
    for index, item in enumerate(
        documents_list,
        start=1,
    ):
        blocks.append(
            f"[{index}]\n"
            f"TITLE: {item['title']}\n"
            f"TEXT: {item['text']}\n"
            f"SOURCE: {item['source']}"
        )
    return "\n\n".join(blocks)


def build_generation_prompt(
    context: dict[str, Any],
) -> str:
    schema = PresentationPlan.model_json_schema()

    return f"""
Отредактируй и структурируй уже подготовленный
материал для внутренней клиентской презентации.

Не выбирай продукты самостоятельно.
Список продуктов уже сформирован поисковой системой.
Нужно только кратко объяснить, с какой задачей клиента
может быть связана каждая тема для обсуждения.

CLIENT:
{context['client_name']}

CLIENT PROFILE:
{context['profile']}

INTERNAL NOTES:
{format_documents(context['notes'])}

ПРЕДВАРИТЕЛЬНО ОТОБРАННЫЕ ТЕМЫ ДЛЯ ОБСУЖДЕНИЯ:
{format_documents(context['products'])}

CLIENT NEWS:
{format_documents(context['news'])}

Требования к содержанию:
- profile_summary: 2–3 предложения;
- financial_tasks: 3–5 пунктов;
- risks: 3–4 риска;
- products: используй до 4 переданных продуктовых тем. Не оценивай их как готовую рекомендацию. только из ПРЕДВАРИТЕЛЬНО ОТОБРАННЫЕ ТЕМЫ ДЛЯ ОБСУЖДЕНИЯ;
- news: максимум 4 новости, только из CLIENT NEWS;
- meeting_questions: 4–6 конкретных вопросов клиенту;
- next_steps: 3–4 практических шага;
- заголовки новостей и источники переписывай дословно.

JSON должен соответствовать этой схеме:
{json.dumps(schema, ensure_ascii=False)}
""".strip()


def extract_json_object(text: str) -> dict[str, Any]:
    cleaned = text.strip()

    cleaned = re.sub(
        r"^```(?:json)?\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\s*```$",
        "",
        cleaned,
    )

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        pass

    start = cleaned.find("{")
    end = cleaned.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(
            "GigaChat не вернул JSON-объект."
        )

    return json.loads(cleaned[start:end + 1])


def normalize_name(value: str) -> str:
    return normalize_text(value)


def ground_plan(
    plan: PresentationPlan,
    context: dict[str, Any],
) -> PresentationPlan:
    plan.client_name = context["client_name"]

    allowed_products = {
        normalize_name(item["title"]): item
        for item in context["products"]
    }

    grounded_products = []
    used_product_names = set()

    for recommendation in plan.products:
        key = normalize_name(recommendation.name)

        if key not in allowed_products:
            close = get_close_matches(
                key,
                allowed_products.keys(),
                n=1,
                cutoff=0.75,
            )
            if not close:
                continue
            key = close[0]

        exact_name = allowed_products[key]["title"]

        if exact_name in used_product_names:
            continue

        recommendation.name = exact_name
        grounded_products.append(recommendation)
        used_product_names.add(exact_name)

    if len(grounded_products) < 3:
        for item in context["products"]:
            if item["title"] in used_product_names:
                continue

            grounded_products.append(
                ProductRecommendation(
                    name=item["title"],
                    fit="Требует подтверждения",
                    rationale=(
                        "Продукт найден RAG-поиском как "
                        "потенциально связанный с задачами клиента."
                    ),
                    caveat=(
                        "Требуется подтверждение потребности, "
                        "срока, объема и риск-профиля."
                    ),
                )
            )
            used_product_names.add(item["title"])

            if len(grounded_products) >= 3:
                break

    plan.products = grounded_products[:4]

    allowed_news = {
        normalize_name(item["title"]): item
        for item in context["news"]
    }

    grounded_news = []
    used_news_titles = set()

    for generated_news in plan.news:
        key = normalize_name(generated_news.title)

        if key not in allowed_news:
            close = get_close_matches(
                key,
                allowed_news.keys(),
                n=1,
                cutoff=0.78,
            )
            if not close:
                continue
            key = close[0]

        original = allowed_news[key]

        if original["title"] in used_news_titles:
            continue

        generated_news.title = original["title"]
        generated_news.source = original["source"]
        grounded_news.append(generated_news)
        used_news_titles.add(original["title"])

    plan.news = grounded_news[:4]

    return plan

pokaskrit='''
def generate_presentation_plan(
    context: dict[str, Any],
) -> PresentationPlan:
    prompt = build_generation_prompt(context)
    raw_response = giga.generate(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=prompt,
        temperature=0.1,
    )

    parsed = extract_json_object(raw_response)

    try:
        plan = PresentationPlan.model_validate(parsed)
    except ValidationError as error:
        print("Ответ модели не прошел валидацию.")
        print(raw_response)
        raise error

    return ground_plan(plan, context)
'''


def generate_presentation_plan(
    context: dict[str, Any],
) -> PresentationPlan:
    prompt = build_generation_prompt(context)

    raw_response = giga.generate(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=prompt,
        temperature=0.0,
    )


    GIGACHAT_BLOCK_MARKERS = [
        "чувствительными темами",
        "ответы на вопросы",
        "временно ограничены",
        "благодарим за понимание",
    ]
    
    
    def is_gigachat_blocked(
        text: str,
    ) -> bool:
        normalized = normalize_text(text)
    
        matches = sum(
            marker in normalized
            for marker in GIGACHAT_BLOCK_MARKERS
        )
    
        return matches >= 2

    if is_gigachat_blocked(raw_response):
        print(
            "GigaChat заблокировал запрос "
            "из-за содержимого контекста."
        )
    
        print(
            "Повторяем без новостей "
            "и с нейтральной формулировкой."
        )
    
        safe_context = {
            **context,
            "news": [],
        }
    
        safe_prompt = build_generation_prompt(
            safe_context
        )
    
        raw_response = giga.generate(
            system_prompt=SYSTEM_PROMPT,
            user_prompt=safe_prompt,
            temperature=0.0,
        )


    
    print("=" * 80)
    print("СЫРОЙ ОТВЕТ GIGACHAT:")
    print("=" * 80)
    print(repr(raw_response))
    print("=" * 80)

    parsed = extract_json_object(raw_response)
    plan = PresentationPlan.model_validate(parsed)

    return ground_plan(plan, context)

# Пример:
# plan = generate_presentation_plan(context)
# display(plan.model_dump())


## 12. Сборка настоящего `.pptx`

In [35]:
# Цветовая палитра
GREEN = RGBColor(18, 120, 85)
DARK_GREEN = RGBColor(8, 64, 49)
LIGHT_GREEN = RGBColor(226, 244, 237)
PALE_GREEN = RGBColor(242, 249, 246)
WHITE = RGBColor(255, 255, 255)
BLACK = RGBColor(28, 34, 32)
DARK_GRAY = RGBColor(73, 82, 79)
MID_GRAY = RGBColor(132, 143, 139)
LIGHT_GRAY = RGBColor(235, 239, 237)
AMBER = RGBColor(211, 145, 43)
FONT_NAME = "Aptos"


def set_slide_background(
    slide,
    color: RGBColor,
) -> None:
    fill = slide.background.fill
    fill.solid()
    fill.fore_color.rgb = color


def shorten_for_slide(
    text: str,
    max_chars: int,
) -> str:
    """
    Ограничивает длину текста без разрыва слова.
    """
    text = re.sub(
        r"\s+",
        " ",
        str(text),
    ).strip()

    if len(text) <= max_chars:
        return text

    shortened = text[:max_chars].rsplit(
        " ",
        1,
    )[0]

    return shortened.rstrip(".,;:—- ") + "…"


def add_textbox(
    slide,
    text: str,
    left: float,
    top: float,
    width: float,
    height: float,
    font_size: float = 18,
    bold: bool = False,
    color: RGBColor = BLACK,
    align: PP_ALIGN = PP_ALIGN.LEFT,
    valign: MSO_ANCHOR = MSO_ANCHOR.TOP,
    margin: float = 0.08,
    auto_fit: bool = False,
    max_chars: int | None = None,
):
    if max_chars is not None:
        text = shorten_for_slide(
            text,
            max_chars=max_chars,
        )

    shape = slide.shapes.add_textbox(
        Inches(left),
        Inches(top),
        Inches(width),
        Inches(height),
    )

    text_frame = shape.text_frame
    text_frame.clear()
    text_frame.word_wrap = True
    text_frame.vertical_anchor = valign

    text_frame.margin_left = Inches(margin)
    text_frame.margin_right = Inches(margin)
    text_frame.margin_top = Inches(margin)
    text_frame.margin_bottom = Inches(margin)

    if auto_fit:
        text_frame.auto_size = (
            MSO_AUTO_SIZE.TEXT_TO_FIT_SHAPE
        )

    paragraph = text_frame.paragraphs[0]
    paragraph.text = str(text)
    paragraph.alignment = align

    paragraph.font.name = FONT_NAME
    paragraph.font.size = Pt(font_size)
    paragraph.font.bold = bold
    paragraph.font.color.rgb = color

    return shape


def add_bullets(
    slide,
    items: list[str],
    left: float,
    top: float,
    width: float,
    height: float,
    font_size: float = 17,
    color: RGBColor = BLACK,
    max_items: int | None = None,
    max_chars_per_item: int | None = None,
    auto_fit: bool = False,
):
    if max_items is not None:
        items = items[:max_items]

    if max_chars_per_item is not None:
        items = [
            shorten_for_slide(
                item,
                max_chars=max_chars_per_item,
            )
            for item in items
        ]

    shape = slide.shapes.add_textbox(
        Inches(left),
        Inches(top),
        Inches(width),
        Inches(height),
    )

    text_frame = shape.text_frame
    text_frame.clear()
    text_frame.word_wrap = True

    text_frame.margin_left = Inches(0.08)
    text_frame.margin_right = Inches(0.05)
    text_frame.margin_top = Inches(0.03)
    text_frame.margin_bottom = Inches(0.03)

    if auto_fit:
        text_frame.auto_size = (
            MSO_AUTO_SIZE.TEXT_TO_FIT_SHAPE
        )

    for index, item in enumerate(items):
        paragraph = (
            text_frame.paragraphs[0]
            if index == 0
            else text_frame.add_paragraph()
        )

        paragraph.text = f"• {item}"
        paragraph.font.name = FONT_NAME
        paragraph.font.size = Pt(font_size)
        paragraph.font.color.rgb = color
        paragraph.space_after = Pt(6)

    return shape


def add_rect(
    slide,
    left: float,
    top: float,
    width: float,
    height: float,
    fill_color: RGBColor,
    line_color: RGBColor | None = None,
    radius: bool = True,
):
    shape_type = (
        MSO_SHAPE.ROUNDED_RECTANGLE
        if radius
        else MSO_SHAPE.RECTANGLE
    )

    shape = slide.shapes.add_shape(
        shape_type,
        Inches(left),
        Inches(top),
        Inches(width),
        Inches(height),
    )
    shape.fill.solid()
    shape.fill.fore_color.rgb = fill_color

    if line_color is None:
        shape.line.fill.background()
    else:
        shape.line.color.rgb = line_color

    return shape


def add_header(
    slide,
    title: str,
    slide_number: int,
) -> None:
    add_textbox(
        slide,
        title,
        0.65,
        0.35,
        11.6,
        0.55,
        font_size=25,
        bold=True,
        color=DARK_GREEN,
    )

    add_textbox(
        slide,
        f"{slide_number:02d}",
        12.2,
        0.38,
        0.45,
        0.35,
        font_size=11,
        bold=True,
        color=MID_GRAY,
        align=PP_ALIGN.RIGHT,
    )

    line = add_rect(
        slide,
        0.65,
        0.98,
        12.0,
        0.025,
        GREEN,
        radius=False,
    )
    line.line.fill.background()


def add_footer(
    slide,
    client_name: str,
) -> None:
    add_textbox(
        slide,
        f"{client_name} · Материалы к клиентской встрече",
        0.65,
        7.12,
        8.0,
        0.2,
        font_size=8.5,
        color=MID_GRAY,
        valign=MSO_ANCHOR.MIDDLE,
    )
    add_textbox(
        slide,
        "Сформировано на основе локальной базы данных",
        9.1,
        7.12,
        3.55,
        0.2,
        font_size=8.5,
        color=MID_GRAY,
        align=PP_ALIGN.RIGHT,
        valign=MSO_ANCHOR.MIDDLE,
    )


def add_card_title(
    slide,
    title: str,
    left: float,
    top: float,
    width: float,
    color: RGBColor = DARK_GREEN,
    font_size: float = 16,
):
    add_textbox(
        slide,
        title,
        left,
        top,
        width,
        0.42,
        font_size=font_size,
        bold=True,
        color=color,
        valign=MSO_ANCHOR.MIDDLE,
    )


def safe_filename(value: str) -> str:
    value = re.sub(
        r"[^0-9A-Za-zА-Яа-я_-]+",
        "_",
        value,
    )
    return value.strip("_") or "client"


def create_pptx(
    plan: PresentationPlan,
    output_path: Path,
) -> Path:
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)
    blank_layout = prs.slide_layouts[6]

    # Слайд 1
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, DARK_GREEN)

    add_rect(
        slide,
        0,
        0,
        13.333,
        0.16,
        GREEN,
        radius=False,
    )

    add_textbox(
        slide,
        "КЛИЕНТСКАЯ ПРЕЗЕНТАЦИЯ",
        0.85,
        1.05,
        5.0,
        0.35,
        font_size=12,
        bold=True,
        color=RGBColor(170, 224, 203),
    )
    add_textbox(
        slide,
        plan.client_name,
        0.8,
        1.65,
        11.5,
        1.15,
        font_size=42,
        bold=True,
        color=WHITE,
        valign=MSO_ANCHOR.MIDDLE,
    )
    add_textbox(
        slide,
        plan.subtitle,
        0.85,
        2.95,
        10.7,
        0.9,
        font_size=20,
        color=RGBColor(225, 238, 233),
    )

    goal_text = (
    "Обсудить ключевые риски, график финансирования, "
    "ликвидность и продуктовые темы для дальнейшей "
    "проработки."
    )

    add_rect(
        slide,
        0.85,
        4.42,
        5.15,
        1.48,
        RGBColor(25, 93, 72),
        line_color=RGBColor(57, 139, 108),
    )
    
    add_textbox(
        slide,
        "Цель встречи",
        1.08,
        4.62,
        4.55,
        0.28,
        font_size=11.5,
        bold=True,
        color=RGBColor(170, 224, 203),
        valign=MSO_ANCHOR.MIDDLE,
    )
    

    add_textbox(
        slide,
        "Обсудить риски, ликвидность и "
        "приоритетные финансовые решения",
        1.05,
        5.08,
        3.55,
        0.55,
        font_size=15,
        color=WHITE,
    )
    add_textbox(
        slide,
        "Конфиденциально · Рабочий материал",
        0.85,
        6.95,
        4.5,
        0.25,
        font_size=9,
        color=RGBColor(165, 190, 181),
    )

    # Слайд 2
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, WHITE)
    add_header(slide, "Профиль клиента и финансовые задачи", 2)

    add_rect(
        slide,
        0.65,
        1.28,
        7.65,
        5.5,
        PALE_GREEN,
        line_color=LIGHT_GREEN,
    )
    add_card_title(
        slide,
        "Профиль",
        0.95,
        1.58,
        6.9,
    )
    add_textbox(
        slide,
        plan.profile_summary,
        0.95,
        2.12,
        6.95,
        2.05,
        font_size=20,
        color=BLACK,
    )

    add_textbox(
        slide,
        "Ключевой фокус",
        0.95,
        4.72,
        2.0,
        0.3,
        font_size=11,
        bold=True,
        color=GREEN,
    )
    add_textbox(
        slide,
        plan.subtitle,
        0.95,
        5.08,
        6.85,
        0.95,
        font_size=19,
        bold=True,
        color=DARK_GREEN,
    )

    add_rect(
        slide,
        8.55,
        1.28,
        4.12,
        5.5,
        WHITE,
        line_color=LIGHT_GRAY,
    )
    add_card_title(
        slide,
        "Финансовые задачи",
        8.88,
        1.58,
        3.45,
    )
    add_bullets(
        slide,
        plan.financial_tasks,
        8.85,
        2.15,
        3.45,
        4.2,
        font_size=16.5,
        max_items=5,
    )
    add_footer(slide, plan.client_name)

    # Слайд 3
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, WHITE)
    add_header(slide, "Ключевые финансовые риски", 3)

    risk_positions = [
        (0.65, 1.35),
        (6.78, 1.35),
        (0.65, 4.05),
        (6.78, 4.05),
    ]

    for index, risk in enumerate(plan.risks[:4]):
        left, top = risk_positions[index]
        add_rect(
            slide,
            left,
            top,
            5.9,
            2.35,
            PALE_GREEN if index % 2 == 0 else WHITE,
            line_color=LIGHT_GREEN,
        )
        number_circle = add_rect(
            slide,
            left + 0.22,
            top + 0.2,
            0.55,
            0.55,
            GREEN,
            radius=True,
        )
        number_circle.line.fill.background()
        add_textbox(
            slide,
            f"{index + 1}",
            left + 0.255,
            top + 0.225,
            0.48,
            0.48,
            font_size=18,
            bold=True,
            color=WHITE,
            align=PP_ALIGN.CENTER,
            valign=MSO_ANCHOR.MIDDLE,
            margin=0.0,
        )

        add_textbox(
            slide,
            risk.name,
            left + 0.95,
            top + 0.18,
            4.55,
            0.55,
            font_size=18,
            bold=True,
            color=DARK_GREEN,
            valign=MSO_ANCHOR.MIDDLE,
        )
        add_textbox(
            slide,
            risk.impact,
            left + 0.28,
            top + 0.92,
            5.25,
            1.05,
            font_size=15.5,
            color=DARK_GRAY,
        )

    add_footer(slide, plan.client_name)

    # Слайд 4
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, WHITE)
    add_header(slide, "Приоритетные продуктовые идеи", 4)

    product_positions = [
        (0.65, 1.35),
        (6.78, 1.35),
        (0.65, 4.05),
        (6.78, 4.05),
    ]

    for index, product in enumerate(plan.products[:4]):
        left, top = product_positions[index]
        add_rect(
            slide,
            left,
            top,
            5.9,
            2.35,
            WHITE,
            line_color=LIGHT_GREEN,
        )

        priority_color = (
            GREEN
            if product.fit.lower().startswith("выс")
            else AMBER
        )

        add_textbox(
            slide,
            product.fit.upper(),
            left + 0.28,
            top + 0.2,
            1.25,
            0.28,
            font_size=9.5,
            bold=True,
            color=priority_color,
        )
        add_textbox(
            slide,
            product.name,
            left + 0.28,
            top + 0.53,
            5.2,
            0.42,
            font_size=19,
            bold=True,
            color=DARK_GREEN,
        )
        add_textbox(
            slide,
            product.rationale,
            left + 0.28,
            top + 1.02,
            5.25,
            0.72,
            font_size=14.5,
            color=BLACK,
        )
        add_textbox(
            slide,
            f"Условие: {product.caveat}",
            left + 0.28,
            top + 1.78,
            5.25,
            0.37,
            font_size=10.5,
            color=MID_GRAY,
        )

    add_footer(slide, plan.client_name)

    # Слайд 5
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, WHITE)
    
    add_header(
        slide,
        "Новости и контекст для встречи",
        5,
    )
    
    news_items = plan.news[:4]
    
    if not news_items:
        add_rect(
            slide,
            0.65,
            1.45,
            12.0,
            4.8,
            PALE_GREEN,
            line_color=LIGHT_GREEN,
        )
    
        add_textbox(
            slide,
            (
                "В локальной базе не найдено новостей, "
                "непосредственно относящихся "
                "к выбранному клиенту."
            ),
            1.35,
            2.75,
            10.6,
            1.1,
            font_size=23,
            bold=True,
            color=DARK_GREEN,
            align=PP_ALIGN.CENTER,
            valign=MSO_ANCHOR.MIDDLE,
            auto_fit=True,
        )
    
    else:
        news_count = len(news_items)
    
        cards_top = 1.25
        cards_height = 5.55
        gap = 0.14
    
        card_height = (
            cards_height
            - gap * (news_count - 1)
        ) / news_count
    
        for index, item in enumerate(news_items):
            top = (
                cards_top
                + index * (card_height + gap)
            )
    
            title = shorten_for_slide(
                item.title,
                max_chars=145,
            )
    
            relevance = shorten_for_slide(
                item.relevance,
                max_chars=220,
            )
    
            source = shorten_for_slide(
                item.source,
                max_chars=55,
            )
    
            # Для длинного заголовка резервируем две строки.
            long_title = len(title) > 78
    
            title_height = (
                0.58
                if long_title
                else 0.38
            )
    
            title_font_size = (
                13.5
                if long_title
                else 15.0
            )
    
            add_rect(
                slide,
                0.65,
                top,
                12.0,
                card_height,
                (
                    PALE_GREEN
                    if index % 2 == 0
                    else WHITE
                ),
                line_color=LIGHT_GREEN,
            )
    
            # Заголовок новости
            add_textbox(
                slide,
                title,
                0.92,
                top + 0.13,
                8.15,
                title_height,
                font_size=title_font_size,
                bold=True,
                color=DARK_GREEN,
                auto_fit=True,
                margin=0.02,
            )
    
            # Источник находится в отдельной правой зоне.
            add_textbox(
                slide,
                source,
                9.35,
                top + 0.14,
                2.92,
                0.38,
                font_size=10,
                bold=True,
                color=GREEN,
                align=PP_ALIGN.RIGHT,
                auto_fit=True,
                margin=0.02,
            )
    
            relevance_top = (
                top
                + 0.17
                + title_height
                + 0.04
            )
    
            relevance_height = max(
                0.25,
                card_height
                - (relevance_top - top)
                - 0.11,
            )
    
            # Основной текст начинается только после
            # фактической зоны заголовка.
            add_textbox(
                slide,
                relevance,
                0.92,
                relevance_top,
                10.95,
                relevance_height,
                font_size=12.2,
                color=DARK_GRAY,
                auto_fit=True,
                margin=0.02,
            )
    
    add_footer(
        slide,
        plan.client_name,
    )

    # Слайд 6
    slide = prs.slides.add_slide(blank_layout)
    set_slide_background(slide, DARK_GREEN)
    
    add_textbox(
        slide,
        "Вопросы к клиенту и следующие шаги",
        0.75,
        0.42,
        11.7,
        0.55,
        font_size=26,
        bold=True,
        color=WHITE,
        auto_fit=True,
    )
    
    # Чуть увеличенные карточки.
    add_rect(
        slide,
        0.75,
        1.25,
        6.05,
        5.45,
        RGBColor(18, 81, 62),
        line_color=RGBColor(51, 125, 98),
    )
    
    add_rect(
        slide,
        7.05,
        1.25,
        5.55,
        5.45,
        RGBColor(18, 81, 62),
        line_color=RGBColor(51, 125, 98),
    )
    
    add_card_title(
        slide,
        "Вопросы на встречу",
        1.05,
        1.54,
        5.4,
        color=RGBColor(178, 230, 210),
        font_size=17,
    )
    
    questions = [
        shorten_for_slide(
            question,
            max_chars=145,
        )
        for question in plan.meeting_questions[:5]
    ]
    
    questions_chars = sum(
        len(question)
        for question in questions
    )
    
    question_font_size = (
        14.5
        if questions_chars < 430
        else 12.8
    )
    
    add_bullets(
        slide,
        questions,
        1.02,
        2.04,
        5.38,
        4.15,
        font_size=question_font_size,
        color=WHITE,
        max_items=5,
        max_chars_per_item=145,
        auto_fit=True,
    )
    
    add_card_title(
        slide,
        "Предлагаемые следующие шаги",
        7.37,
        1.54,
        4.9,
        color=RGBColor(178, 230, 210),
        font_size=17,
    )
    
    next_steps = [
        shorten_for_slide(
            step,
            max_chars=130,
        )
        for step in plan.next_steps[:4]
    ]
    
    steps_chars = sum(
        len(step)
        for step in next_steps
    )
    
    steps_font_size = (
        14.5
        if steps_chars < 330
        else 12.8
    )
    
    add_bullets(
        slide,
        next_steps,
        7.35,
        2.04,
        4.85,
        2.85,
        font_size=steps_font_size,
        color=WHITE,
        max_items=4,
        max_chars_per_item=130,
        auto_fit=True,
    )
    
    # Дисклеймер имеет собственную зону,
    # поэтому не пересекается со списком.
    add_rect(
        slide,
        7.34,
        5.18,
        4.92,
        1.05,
        RGBColor(14, 71, 54),
        line_color=RGBColor(45, 111, 87),
    )
    
    add_textbox(
        slide,
        (
            "Финальные параметры обсуждаются после "
            "подтверждения объема, срока, графика "
            "денежных потоков и риск-профиля."
        ),
        7.57,
        5.38,
        4.45,
        0.62,
        font_size=10.5,
        color=RGBColor(184, 207, 199),
        auto_fit=True,
        max_chars=210,
    )
    
    add_textbox(
        slide,
        f"{plan.client_name} · Конфиденциально",
        0.8,
        7.0,
        4.0,
        0.22,
        font_size=9,
        color=RGBColor(165, 190, 181),
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    prs.save(output_path)

    return output_path


## 13. Полный агент: один вызов → `.pptx`

In [15]:
def ensure_index_ready(
    rebuild_index: bool = False,
) -> None:
    index_exists = opensearch.indices.exists(
        index=OPENSEARCH_INDEX,
    )

    if rebuild_index or not index_exists:
        index_documents(
            opensearch,
            documents,
            rebuild_index=True,
        )


def generate_client_presentation(
    client_name: str,
    rebuild_index: bool = False,
) -> tuple[Path, PresentationPlan]:
    ensure_index_ready(
        rebuild_index=rebuild_index,
    )

    context = build_rag_context(
        opensearch,
        source_data,
        client_name,
    )

    print_context(context)

    plan = generate_presentation_plan(context)

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    filename = (
        f"{safe_filename(client_name)}_"
        f"{timestamp}.pptx"
    )
    output_path = OUTPUT_DIR / filename

    create_pptx(
        plan=plan,
        output_path=output_path,
    )

    plan_path = output_path.with_suffix(".json")
    plan_path.write_text(
        json.dumps(
            plan.model_dump(),
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\nГотовая презентация:", output_path)
    print("JSON-план:", plan_path)

    return output_path, plan


## 14. Примеры запуска

In [16]:
# При первом запуске пересоздаем индекс и считаем embeddings:
pptx_path, plan = generate_client_presentation(
    "РЖД",
    rebuild_index=True,
)

pptx_path


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.86s/it]


Пакет 1: проиндексировано 32/1906


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  3.83s/it]


Пакет 2: проиндексировано 64/1906


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.05s/it]


...

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.26s/it]


Пакет 59: проиндексировано 1888/1906


Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.27s/it]


Пакет 60: проиндексировано 1906/1906
Индексация завершена.
КЛИЕНТ: РЖД

ПРОФИЛЬ:
 Инфраструктурная транспортная компания с масштабной инвестиционной программой, длинным горизонтом финансирования и высокой потребностью в управлении долговой нагрузкой. Основные финансовые риски: рост процентных ставок, кассовые разрывы по крупным проектам, валютные платежи по закупкам оборудования, необходимость предсказуемого фондирования. Возможные темы для встречи: IRS, долгосрочное фондирование, Repo, Deposit, валютные форварды.

ЗАМЕТКИ:
- Для РЖД в презентации важны долгий горизонт инвестиций, стоимость фондирования и график платежей. IRS может быть сильной идеей для стабилизации процентных расходов. Repo может подойти для краткосрочного управления ликвидностью, а Deposit — для размещения временных остатков. FX Forward имеет смысл только при наличии валютных закупок оборудования. Commodity Hedge для РЖД обычно не является приоритетным продуктом.

КАНДИДАТЫ-ПРОДУКТЫ:
- Deposit
- IRS
- FX Forward
- D

ValueError: GigaChat не вернул JSON-объект.

In [30]:
 #Второй пример — индекс уже существует:
 pptx_path_3, plan_3 = generate_client_presentation(
     "Лукойл",
     rebuild_index=False,
 )
 pptx_path_3


КЛИЕНТ: Лукойл

ПРОФИЛЬ:
 Нефтегазовая компания с экспортной выручкой, капитальными затратами и чувствительностью к ценам на нефть и нефтепродукты. Основные финансовые риски: товарная волатильность, валютные денежные потоки, процентный риск по долговому портфелю, управление ликвидностью в периоды крупных налоговых и инвестиционных платежей. Возможные темы для встречи: Commodity Hedge, FX Forward, XCCY, IRS, краткосрочное размещение ликвидности.

ЗАМЕТКИ:

КАНДИДАТЫ-ПРОДУКТЫ:
- Deposit
- Commodity Hedge
- DCD
- Structured Note
- IRS
- Repo
- FX Forward

НОВОСТИ:
СЫРОЙ ОТВЕТ GIGACHAT:
'{\n  "client_name": "Лукойл",\n  "subtitle": "Структура финансовой сессии",\n  "profile_summary": "Крупная нефтегазовая компания с высокой зависимостью прибыли от мировых цен на нефть и нефтепродукты. Ключевые финансовые задачи включают защиту от ценовой волатильности сырья, эффективное управление валютными потоками и оптимизацию структуры долга.",\n  "financial_tasks": [\n    "Хеджирование рисков изменени

WindowsPath('D:/sber/outputs/Лукойл_20260720_190530.pptx')

In [42]:

# ============================================================
# ЗАДАНИЕ 3. СИСТЕМА ВАЛИДАЦИИ АГЕНТА
# ============================================================

from __future__ import annotations

import json
import math
import random
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from pptx import Presentation


EVAL_DIR = PROJECT_DIR / "evaluation"
EVAL_RUNS_DIR = EVAL_DIR / "runs"
EVAL_COMPARISONS_DIR = EVAL_DIR / "comparisons"

EVAL_RUNS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_COMPARISONS_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# 1. Тестовый набор
# ------------------------------------------------------------

GOLD_CLIENT_SPECS: dict[str, dict[str, Any]] = {
    "РусГидро": {
        "split": "dev",
        "core_products": ["IRS"],
        "acceptable_products": [
            "FX Forward",
            "Deposit",
            "Structured Note",
        ],
        "forbidden_products": ["Commodity Hedge"],
        "risk_keyword_groups": [
            ["процент", "ставк", "долг"],
            ["инвест", "капитальн", "проект"],
            ["валют", "оборудован"],
            ["ликвид", "кассов"],
        ],
    },
    "Аэрофлот": {
        "split": "dev",
        "core_products": [
            "FX Forward",
            "Option Collar",
            "Commodity Hedge",
        ],
        "acceptable_products": [
            "Deposit",
            "Repo",
        ],
        "forbidden_products": [
            "DCD",
            "Structured Note",
        ],
        "risk_keyword_groups": [
            ["валют", "курс"],
            ["топлив", "товар"],
            ["сезон", "ликвид", "кассов"],
        ],
    },
    "Норникель": {
        "split": "holdout",
        "core_products": [
            "Commodity Hedge",
            "FX Forward",
            "Option Collar",
        ],
        "acceptable_products": [
            "IRS",
            "Deposit",
            "Prepaid Forward",
        ],
        "forbidden_products": [],
        "risk_keyword_groups": [
            ["товар", "металл", "никел", "мед", "паллад"],
            ["валют", "экспорт"],
            ["инвест", "эколог", "капитальн"],
            ["ликвид"],
        ],
    },
    "РЖД": {
        "split": "dev",
        "core_products": [
            "IRS",
            "Repo",
            "Deposit",
        ],
        "acceptable_products": ["FX Forward"],
        "forbidden_products": ["Commodity Hedge"],
        "risk_keyword_groups": [
            ["процент", "ставк", "долг", "фондирован"],
            ["инвест", "проект", "капитальн"],
            ["ликвид", "кассов", "платеж"],
            ["валют", "оборудован"],
        ],
    },
    "Магнит": {
        "split": "dev",
        "core_products": [
            "Deposit",
            "Repo",
            "FX Forward",
        ],
        "acceptable_products": ["Structured Note"],
        "forbidden_products": ["Commodity Hedge"],
        "risk_keyword_groups": [
            ["оборотн", "ликвид", "поставщик"],
            ["фондирован", "ставк"],
            ["валют", "импорт"],
            ["остат", "размещен"],
        ],
    },
    "Лукойл": {
        "split": "dev",
        "core_products": [
            "Commodity Hedge",
            "FX Forward",
            "XCCY",
        ],
        "acceptable_products": [
            "IRS",
            "Deposit",
            "Repo",
        ],
        "forbidden_products": [],
        "risk_keyword_groups": [
            ["нефт", "товар", "цен"],
            ["валют", "экспорт"],
            ["процент", "ставк", "долг"],
            ["ликвид", "налог", "инвест"],
        ],
    },
    "МТС": {
        "split": "dev",
        "core_products": [
            "IRS",
            "FX Forward",
            "Deposit",
        ],
        "acceptable_products": ["Structured Note"],
        "forbidden_products": ["Commodity Hedge"],
        "risk_keyword_groups": [
            ["процент", "ставк", "долг"],
            ["валют", "оборудован"],
            ["ликвид", "остат"],
            ["инвест", "сет", "цифров"],
        ],
    },
    "ФосАгро": {
        "split": "holdout",
        "core_products": [
            "FX Forward",
            "Option Collar",
            "Commodity Hedge",
        ],
        "acceptable_products": [
            "IRS",
            "DCD",
            "Deposit",
        ],
        "forbidden_products": ["Structured Note"],
        "risk_keyword_groups": [
            ["валют", "экспорт"],
            ["товар", "сырь", "газ", "логист"],
            ["сезон", "оборотн", "ликвид"],
            ["процент", "ставк", "долг"],
        ],
    },
}


def save_gold_test_set() -> Path:
    path = EVAL_DIR / "gold_test_set.json"
    path.write_text(
        json.dumps(
            GOLD_CLIENT_SPECS,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return path


GOLD_TEST_SET_PATH = save_gold_test_set()
print("Gold test set:", GOLD_TEST_SET_PATH)


# ------------------------------------------------------------
# 2. Конфигурация версии агента
# ------------------------------------------------------------

@dataclass(frozen=True)
class AgentVersion:
    name: str
    description: str

    product_top_k: int = 7
    news_candidate_k: int = 12

    direct_news_only: bool = True
    safe_news_only: bool = True

    prompt_variant: str = "neutral_v2"

    max_products_in_plan: int = 4
    max_news_in_plan: int = 4

    # Можно передать отдельную функцию сборки PPTX
    # для сравнения layout-версий.
    pptx_builder: Callable | None = field(
        default=None,
        compare=False,
        repr=False,
    )

    def serializable_dict(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "product_top_k": self.product_top_k,
            "news_candidate_k": self.news_candidate_k,
            "direct_news_only": self.direct_news_only,
            "safe_news_only": self.safe_news_only,
            "prompt_variant": self.prompt_variant,
            "max_products_in_plan": self.max_products_in_plan,
            "max_news_in_plan": self.max_news_in_plan,
            "pptx_builder": (
                self.pptx_builder.__name__
                if self.pptx_builder is not None
                else "create_pptx"
            ),
        }


BASELINE_VERSION = AgentVersion(
    name="baseline_v1",
    description=(
        "Широкий новостной retrieval, исходная финансовая "
        "формулировка промпта."
    ),
    product_top_k=7,
    news_candidate_k=8,
    direct_news_only=False,
    safe_news_only=False,
    prompt_variant="baseline_v1",
)

CANDIDATE_VERSION = AgentVersion(
    name="candidate_v2",
    description=(
        "Прямые безопасные новости, нейтральный промпт "
        "технического редактора."
    ),
    product_top_k=6,
    news_candidate_k=20,
    direct_news_only=True,
    safe_news_only=True,
    prompt_variant="neutral_v2",
)


# ------------------------------------------------------------
# 3. Вспомогательные функции для новостей
# ------------------------------------------------------------

EVAL_SENSITIVE_NEWS_PATTERNS = [
    r"\bфронт\w*",
    r"\bвоенн\w*",
    r"\bоруж\w*",
    r"\bвзрыв\w*",
    r"\bракет\w*",
    r"\bбпла\b",
    r"\bдрон\w*",
    r"\bбоеприпас\w*",
    r"\bогнев\w*",
    r"\bубийств\w*",
    r"\bтеррор\w*",
    r"\bмобилизац\w*",
    r"\bвойн\w*",
]


def eval_normalize_text(value: str) -> str:
    if "normalize_text" in globals():
        return normalize_text(value)

    value = str(value).lower().replace("ё", "е")
    value = re.sub(r"[^a-zа-я0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def eval_contains_client_alias(
    text: str,
    client_name: str,
) -> bool:
    normalized_text = f" {eval_normalize_text(text)} "

    aliases = CLIENT_ALIASES.get(
        client_name,
        [client_name],
    )

    aliases = list(aliases) + [client_name]

    for alias in aliases:
        normalized_alias = eval_normalize_text(alias)

        if (
            normalized_alias
            and f" {normalized_alias} "
            in normalized_text
        ):
            return True

    return False


def eval_is_sensitive_news(
    title: str,
    text: str,
) -> bool:
    combined = eval_normalize_text(
        f"{title} {text}"
    )

    return any(
        re.search(
            pattern,
            combined,
            flags=re.IGNORECASE,
        )
        for pattern in EVAL_SENSITIVE_NEWS_PATTERNS
    )


def build_direct_news_gold(
    client_name: str,
) -> list[dict[str, str]]:
    gold_news = []

    for _, row in source_data["news"].iterrows():
        title = str(row["title"]).strip()
        text = str(row["text"]).strip()
        source = str(row["source"]).strip()

        if not eval_contains_client_alias(
            title,
            client_name,
        ):
            continue

        if eval_is_sensitive_news(title, text):
            continue

        gold_news.append({
            "title": title,
            "source": source,
        })

    # Удаляем дубликаты по нормализованному заголовку.
    unique = {}
    for item in gold_news:
        key = eval_normalize_text(item["title"])
        unique[key] = item

    return list(unique.values())


# ------------------------------------------------------------
# 4. Версионированный retrieval
# ------------------------------------------------------------

def build_eval_context(
    version: AgentVersion,
    client_name: str,
) -> dict[str, Any]:
    profile = get_client_profile(
        source_data,
        client_name,
    )

    note_query = (
        f"{client_name}. Внутренние рекомендации, "
        "приоритеты и ограничения."
    )

    notes = hybrid_search(
        client=opensearch,
        query_text=note_query,
        doc_type="internal_note",
        client_name=client_name,
        top_k=2,
    )

    note_text = "\n".join(
        item["text"]
        for item in notes
    )

    product_query = (
        f"Клиент: {client_name}.\n"
        f"Профиль: {profile}\n"
        f"Внутренние рекомендации: {note_text}\n"
        "Нужно найти финансовые продукты, связанные "
        "с рисками, фондированием и ликвидностью."
    )

    products = hybrid_search(
        client=opensearch,
        query_text=product_query,
        doc_type="product",
        top_k=version.product_top_k,
    )

    news_query = (
        f"{client_name}: корпоративные новости, "
        "финансирование, инвестиции, долговая нагрузка, "
        "ликвидность и операционные события"
    )

    raw_news = hybrid_search(
        client=opensearch,
        query_text=news_query,
        doc_type="news",
        client_name=client_name,
        top_k=version.news_candidate_k,
    )

    filtered_news = []

    for item in raw_news:
        if (
            version.direct_news_only
            and not eval_contains_client_alias(
                item["title"],
                client_name,
            )
        ):
            continue

        if (
            version.safe_news_only
            and eval_is_sensitive_news(
                item["title"],
                item["text"],
            )
        ):
            continue

        filtered_news.append(item)

        if len(filtered_news) >= version.max_news_in_plan:
            break

    return {
        "client_name": client_name,
        "profile": profile,
        "notes": notes,
        "products": products,
        "news": filtered_news,
    }


# ------------------------------------------------------------
# 5. Две версии промпта
# ------------------------------------------------------------

EVAL_BASELINE_SYSTEM_PROMPT = """
Ты — корпоративный банкир и аналитик.
Подготовь план презентации для встречи с клиентом.

Используй только предоставленный контекст.
Подбери наиболее подходящие финансовые продукты.
Верни только JSON.
""".strip()


EVAL_NEUTRAL_SYSTEM_PROMPT = """
Ты — технический редактор внутренних корпоративных
презентаций.

Тебе уже передан подготовленный набор фактов,
кандидатов-продуктов и новостей.

Ты не принимаешь финансовых решений.
Ты только:
- сокращаешь переданный текст;
- структурируешь его для слайдов;
- сохраняешь ограничения;
- не добавляешь внешние сведения.

Продукты называй темами для обсуждения,
а не готовыми рекомендациями.

Верни только один валидный JSON-объект.
Первый символ ответа должен быть {
Последний символ ответа должен быть }
Не используй Markdown.
""".strip()


def build_eval_prompt(
    context: dict[str, Any],
    version: AgentVersion,
) -> tuple[str, str]:
    allowed_products = [
        item["title"]
        for item in context["products"]
    ]

    allowed_news = [
        {
            "title": item["title"],
            "source": item["source"],
        }
        for item in context["news"]
    ]

    structure = {
        "client_name": context["client_name"],
        "subtitle": "Краткий фокус встречи",
        "profile_summary": "2–3 предложения",
        "financial_tasks": [
            "Задача 1",
            "Задача 2",
            "Задача 3",
        ],
        "risks": [
            {
                "name": "Название риска",
                "impact": "Почему риск важен",
            }
        ],
        "products": [
            {
                "name": "Точное название из списка",
                "fit": "Основная тема",
                "rationale": "Связь с задачей клиента",
                "caveat": "Что нужно подтвердить",
            }
        ],
        "news": [
            {
                "title": "Точный заголовок",
                "source": "Точный источник",
                "relevance": "Связь с клиентской встречей",
            }
        ],
        "meeting_questions": [
            "Вопрос 1",
            "Вопрос 2",
            "Вопрос 3",
            "Вопрос 4",
        ],
        "next_steps": [
            "Шаг 1",
            "Шаг 2",
            "Шаг 3",
        ],
    }

    if version.prompt_variant == "baseline_v1":
        system_prompt = EVAL_BASELINE_SYSTEM_PROMPT

        user_prompt = f"""
Подготовь презентацию о клиенте {context["client_name"]}.

ПРОФИЛЬ:
{context["profile"]}

ВНУТРЕННИЕ ЗАМЕТКИ:
{format_documents(context["notes"])}

ПРОДУКТЫ:
{format_documents(context["products"])}

НОВОСТИ:
{format_documents(context["news"])}

Верни JSON по структуре:
{json.dumps(
    structure,
    ensure_ascii=False,
    indent=2,
)}
""".strip()

        return system_prompt, user_prompt

    system_prompt = EVAL_NEUTRAL_SYSTEM_PROMPT

    user_prompt = f"""
Отредактируй и структурируй уже подготовленный
материал для внутренней клиентской презентации.

КЛИЕНТ:
{context["client_name"]}

ПРОФИЛЬ:
{context["profile"]}

ВНУТРЕННИЕ ЗАМЕТКИ:
{format_documents(context["notes"])}

ПРЕДВАРИТЕЛЬНО ОТОБРАННЫЕ ТЕМЫ:
{format_documents(context["products"])}

РАЗРЕШЕННЫЕ НАЗВАНИЯ ПРОДУКТОВ:
{json.dumps(
    allowed_products,
    ensure_ascii=False,
)}

РАЗРЕШЕННЫЕ НОВОСТИ:
{json.dumps(
    allowed_news,
    ensure_ascii=False,
    indent=2,
)}

ПРАВИЛА:
1. Не выбирай продукты вне переданного списка.
2. Используй не более {version.max_products_in_plan} продуктов.
3. Используй не более {version.max_news_in_plan} новостей.
4. Заголовки и источники новостей копируй точно.
5. Если новостей нет, news должен быть пустым списком.
6. Не придумывай показатели и сделки.
7. Верни только JSON без Markdown.

СТРУКТУРА:
{json.dumps(
    structure,
    ensure_ascii=False,
    indent=2,
)}
""".strip()

    return system_prompt, user_prompt


# ------------------------------------------------------------
# 6. Генерация с диагностикой
# ------------------------------------------------------------

EVAL_BLOCK_MARKERS = [
    "чувствительными темами",
    "временно ограничены",
    "благодарим за понимание",
]


def eval_is_blocked_response(
    text: str,
) -> bool:
    normalized = eval_normalize_text(text)

    return sum(
        marker in normalized
        for marker in EVAL_BLOCK_MARKERS
    ) >= 2


@dataclass
class GenerationDiagnostics:
    mode: str
    attempts: int
    blocked: bool
    parse_error: str | None
    raw_response: str
    latency_seconds: float


def build_eval_fallback_plan(
    context: dict[str, Any],
) -> PresentationPlan:
    products = []

    for index, item in enumerate(
        context["products"][:4]
    ):
        if index == 0:
            fit = "Основная тема"
        elif index == 1:
            fit = "Дополнительная тема"
        else:
            fit = "Требует подтверждения"

        products.append(
            ProductRecommendation(
                name=item["title"],
                fit=fit,
                rationale=(
                    str(item["text"])[:260]
                ),
                caveat=(
                    "Нужно подтвердить задачу, срок, объем "
                    "и график денежных потоков."
                ),
            )
        )

    risks = [
        RiskItem(
            name="Стоимость фондирования",
            impact=(
                "Условия привлечения финансирования "
                "могут влиять на стоимость реализации задач."
            ),
        ),
        RiskItem(
            name="Риск ликвидности",
            impact=(
                "Нужно сопоставить сроки поступлений "
                "и будущих платежей."
            ),
        ),
        RiskItem(
            name="Рыночные риски",
            impact=(
                "Ставки, курсы или товарные цены могут "
                "изменять денежные потоки клиента."
            ),
        ),
    ]

    news = [
        NewsItem(
            title=item["title"],
            source=item["source"],
            relevance=(
                "Новость непосредственно относится "
                "к клиенту и может быть контекстом встречи."
            ),
        )
        for item in context["news"][:4]
    ]

    return PresentationPlan(
        client_name=context["client_name"],
        subtitle=(
            "Обсуждение рисков, финансирования "
            "и ликвидности"
        ),
        profile_summary=str(context["profile"])[:650],
        financial_tasks=[
            "Уточнить график финансирования и платежей.",
            "Определить приоритетные финансовые риски.",
            "Согласовать горизонт управления ликвидностью.",
            "Подтвердить параметры продуктовых тем.",
        ],
        risks=risks,
        products=products,
        news=news,
        meeting_questions=[
            "Каков актуальный график финансирования?",
            "Какие риски требуют первоочередного внимания?",
            "Есть ли подтвержденные валютные потоки?",
            "Каков горизонт свободной ликвидности?",
        ],
        next_steps=[
            "Получить график денежных потоков.",
            "Уточнить объемы и сроки.",
            "Подготовить предварительные параметры.",
        ],
    )


def generate_eval_plan(
    context: dict[str, Any],
    version: AgentVersion,
) -> tuple[PresentationPlan, GenerationDiagnostics]:
    system_prompt, user_prompt = build_eval_prompt(
        context,
        version,
    )

    started = time.perf_counter()
    raw_response = ""
    parse_error = None
    blocked = False

    for attempt in range(1, 3):
        try:
            raw_response = giga.generate(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                temperature=0.0,
            )
        except Exception as error:
            parse_error = (
                f"API error: {type(error).__name__}: {error}"
            )
            break

        if eval_is_blocked_response(raw_response):
            blocked = True

            # Вторая попытка: без новостей и с нейтральным промптом.
            if attempt == 1:
                safe_context = {
                    **context,
                    "news": [],
                }
                safe_version = AgentVersion(
                    **{
                        **version.serializable_dict(),
                        "prompt_variant": "neutral_v2",
                        "pptx_builder": version.pptx_builder,
                    }
                )
                system_prompt, user_prompt = build_eval_prompt(
                    safe_context,
                    safe_version,
                )
                continue

            break

        try:
            parsed = extract_json_object(raw_response)
            plan = PresentationPlan.model_validate(parsed)
            plan = ground_plan(plan, context)

            latency = time.perf_counter() - started

            return plan, GenerationDiagnostics(
                mode=(
                    "llm"
                    if attempt == 1
                    else "repair"
                ),
                attempts=attempt,
                blocked=blocked,
                parse_error=None,
                raw_response=raw_response,
                latency_seconds=latency,
            )

        except Exception as error:
            parse_error = (
                f"{type(error).__name__}: {error}"
            )

            if attempt == 1:
                user_prompt = f"""
Исправь предыдущий ответ и верни только валидный JSON.
Не добавляй пояснений и Markdown.

Предыдущий ответ:
{raw_response[:5000]}

Обязательные поля:
client_name, subtitle, profile_summary,
financial_tasks, risks, products, news,
meeting_questions, next_steps.
""".strip()
                system_prompt = EVAL_NEUTRAL_SYSTEM_PROMPT
                continue

    fallback = build_eval_fallback_plan(context)
    fallback = ground_plan(fallback, context)

    latency = time.perf_counter() - started

    return fallback, GenerationDiagnostics(
        mode="fallback",
        attempts=2,
        blocked=blocked,
        parse_error=parse_error,
        raw_response=raw_response,
        latency_seconds=latency,
    )


# ------------------------------------------------------------
# 7. Метрики retrieval и генерации
# ------------------------------------------------------------

def safe_ratio(
    numerator: float,
    denominator: float,
    *,
    empty_value: float = 1.0,
) -> float:
    if denominator == 0:
        return float(empty_value)

    return float(numerator / denominator)


def normalized_set(values: list[str]) -> set[str]:
    return {
        eval_normalize_text(value)
        for value in values
        if str(value).strip()
    }


def product_retrieval_metrics(
    context: dict[str, Any],
    gold: dict[str, Any],
) -> dict[str, float]:
    retrieved = [
        item["title"]
        for item in context["products"]
    ]

    retrieved_set = normalized_set(retrieved)
    core_set = normalized_set(
        gold["core_products"]
    )
    acceptable_set = normalized_set(
        gold["acceptable_products"]
    )
    forbidden_set = normalized_set(
        gold["forbidden_products"]
    )

    relevant_set = core_set | acceptable_set

    relevant_hits = retrieved_set & relevant_set
    core_hits = retrieved_set & core_set
    forbidden_hits = retrieved_set & forbidden_set

    first_core_rank = None

    for rank, product_name in enumerate(
        retrieved,
        start=1,
    ):
        if eval_normalize_text(product_name) in core_set:
            first_core_rank = rank
            break

    mrr = (
        1.0 / first_core_rank
        if first_core_rank is not None
        else 0.0
    )

    return {
        "m_product_precision": safe_ratio(
            len(relevant_hits),
            len(retrieved_set),
            empty_value=0.0,
        ),
        "m_product_core_recall": safe_ratio(
            len(core_hits),
            len(core_set),
            empty_value=1.0,
        ),
        "m_product_forbidden_clean": (
            1.0
            - safe_ratio(
                len(forbidden_hits),
                max(1, len(retrieved_set)),
                empty_value=0.0,
            )
        ),
        "m_product_mrr": mrr,
    }


def news_retrieval_metrics(
    context: dict[str, Any],
    client_name: str,
) -> dict[str, float]:
    gold_news = build_direct_news_gold(
        client_name
    )

    gold_titles = normalized_set([
        item["title"]
        for item in gold_news
    ])

    retrieved_titles = normalized_set([
        item["title"]
        for item in context["news"]
    ])

    hits = gold_titles & retrieved_titles

    if not gold_titles:
        precision = (
            1.0
            if not retrieved_titles
            else 0.0
        )
        recall = precision

    else:
        precision = safe_ratio(
            len(hits),
            len(retrieved_titles),
            empty_value=0.0,
        )
        recall = safe_ratio(
            len(hits),
            len(gold_titles),
            empty_value=0.0,
        )

    return {
        "m_news_precision": precision,
        "m_news_recall": recall,
        "n_gold_news": float(len(gold_titles)),
        "n_retrieved_news": float(
            len(retrieved_titles)
        ),
    }


def risk_keyword_coverage(
    plan: PresentationPlan,
    gold: dict[str, Any],
) -> float:
    combined = eval_normalize_text(
        " ".join(
            [
                risk.name
                + " "
                + risk.impact
                for risk in plan.risks
            ]
        )
    )

    groups = gold["risk_keyword_groups"]

    covered = 0

    for group in groups:
        if any(
            eval_normalize_text(keyword)
            in combined
            for keyword in group
        ):
            covered += 1

    return safe_ratio(
        covered,
        len(groups),
        empty_value=1.0,
    )


def plan_conciseness_score(
    plan: PresentationPlan,
) -> float:
    checks = []

    checks.append(
        len(plan.subtitle) <= 220
    )
    checks.append(
        len(plan.profile_summary) <= 700
    )

    checks.extend(
        len(item) <= 180
        for item in plan.financial_tasks
    )

    checks.extend(
        len(risk.name) <= 90
        and len(risk.impact) <= 280
        for risk in plan.risks
    )

    checks.extend(
        len(product.rationale) <= 320
        and len(product.caveat) <= 240
        for product in plan.products
    )

    checks.extend(
        len(news.title) <= 180
        and len(news.relevance) <= 280
        for news in plan.news
    )

    checks.extend(
        len(item) <= 180
        for item in plan.meeting_questions
    )

    checks.extend(
        len(item) <= 160
        for item in plan.next_steps
    )

    return safe_ratio(
        sum(checks),
        len(checks),
        empty_value=1.0,
    )


def generation_metrics(
    context: dict[str, Any],
    plan: PresentationPlan,
    diagnostics: GenerationDiagnostics,
    gold: dict[str, Any],
) -> dict[str, float]:
    retrieved_products = normalized_set([
        item["title"]
        for item in context["products"]
    ])

    generated_products = normalized_set([
        item.name
        for item in plan.products
    ])

    retrieved_news = normalized_set([
        item["title"]
        for item in context["news"]
    ])

    generated_news = normalized_set([
        item.title
        for item in plan.news
    ])

    core_products = normalized_set(
        gold["core_products"]
    )
    forbidden_products = normalized_set(
        gold["forbidden_products"]
    )

    section_checks = [
        3 <= len(plan.financial_tasks) <= 5,
        3 <= len(plan.risks) <= 4,
        3 <= len(plan.products) <= 4,
        len(plan.news) <= 4,
        4 <= len(plan.meeting_questions) <= 6,
        3 <= len(plan.next_steps) <= 4,
    ]

    product_grounding = safe_ratio(
        len(
            generated_products
            & retrieved_products
        ),
        len(generated_products),
        empty_value=0.0,
    )

    news_grounding = safe_ratio(
        len(
            generated_news
            & retrieved_news
        ),
        len(generated_news),
        empty_value=(
            1.0
            if not generated_news
            else 0.0
        ),
    )

    core_coverage = safe_ratio(
        len(
            generated_products
            & core_products
        ),
        len(core_products),
        empty_value=1.0,
    )

    forbidden_clean = (
        1.0
        - safe_ratio(
            len(
                generated_products
                & forbidden_products
            ),
            max(1, len(generated_products)),
            empty_value=0.0,
        )
    )

    return {
        "m_schema_success": 1.0,
        "m_no_fallback": (
            0.0
            if diagnostics.mode == "fallback"
            else 1.0
        ),
        "m_generation_not_blocked": (
            0.0
            if diagnostics.blocked
            else 1.0
        ),
        "m_section_completeness": safe_ratio(
            sum(section_checks),
            len(section_checks),
            empty_value=1.0,
        ),
        "m_generated_product_grounding": (
            product_grounding
        ),
        "m_generated_core_coverage": core_coverage,
        "m_generated_forbidden_clean": forbidden_clean,
        "m_generated_news_grounding": news_grounding,
        "m_risk_keyword_coverage": (
            risk_keyword_coverage(
                plan,
                gold,
            )
        ),
        "m_plan_conciseness": (
            plan_conciseness_score(plan)
        ),
        "latency_generation_seconds": (
            diagnostics.latency_seconds
        ),
        "generation_attempts": float(
            diagnostics.attempts
        ),
    }


# ------------------------------------------------------------
# 8. Метрики PPTX
# ------------------------------------------------------------

EMU_PER_INCH = 914400


def emu_to_inches(value: int | float) -> float:
    return float(value) / EMU_PER_INCH


def shape_text(shape) -> str:
    if not getattr(
        shape,
        "has_text_frame",
        False,
    ):
        return ""

    return "\n".join(
        paragraph.text
        for paragraph in shape.text_frame.paragraphs
    ).strip()


def shape_font_size_pt(shape) -> float:
    sizes = []

    if not getattr(
        shape,
        "has_text_frame",
        False,
    ):
        return 14.0

    for paragraph in shape.text_frame.paragraphs:
        for run in paragraph.runs:
            if run.font.size is not None:
                sizes.append(run.font.size.pt)

    if sizes:
        return max(8.0, min(sizes))

    return 14.0


def text_shape_rect(shape) -> tuple[float, float, float, float]:
    left = emu_to_inches(shape.left)
    top = emu_to_inches(shape.top)
    width = emu_to_inches(shape.width)
    height = emu_to_inches(shape.height)

    return (
        left,
        top,
        left + width,
        top + height,
    )


def intersection_ratio(
    rect_a: tuple[float, float, float, float],
    rect_b: tuple[float, float, float, float],
    tolerance: float = 0.025,
) -> float:
    a_left, a_top, a_right, a_bottom = rect_a
    b_left, b_top, b_right, b_bottom = rect_b

    # Уменьшаем прямоугольники на небольшую величину,
    # чтобы соприкосновение границ не считалось наложением.
    a_left += tolerance
    a_top += tolerance
    a_right -= tolerance
    a_bottom -= tolerance

    b_left += tolerance
    b_top += tolerance
    b_right -= tolerance
    b_bottom -= tolerance

    overlap_width = max(
        0.0,
        min(a_right, b_right)
        - max(a_left, b_left),
    )

    overlap_height = max(
        0.0,
        min(a_bottom, b_bottom)
        - max(a_top, b_top),
    )

    intersection = (
        overlap_width
        * overlap_height
    )

    area_a = max(
        0.0,
        (a_right - a_left)
        * (a_bottom - a_top),
    )

    area_b = max(
        0.0,
        (b_right - b_left)
        * (b_bottom - b_top),
    )

    min_area = min(area_a, area_b)

    if min_area <= 0:
        return 0.0

    return intersection / min_area


def estimate_text_capacity(shape) -> float:
    text_frame = shape.text_frame

    width = emu_to_inches(shape.width)
    height = emu_to_inches(shape.height)

    margin_left = emu_to_inches(
        text_frame.margin_left or 0
    )
    margin_right = emu_to_inches(
        text_frame.margin_right or 0
    )
    margin_top = emu_to_inches(
        text_frame.margin_top or 0
    )
    margin_bottom = emu_to_inches(
        text_frame.margin_bottom or 0
    )

    usable_width = max(
        0.1,
        width - margin_left - margin_right,
    )

    usable_height = max(
        0.1,
        height - margin_top - margin_bottom,
    )

    font_size = shape_font_size_pt(shape)

    average_char_width = (
        font_size / 72.0 * 0.52
    )

    line_height = (
        font_size / 72.0 * 1.25
    )

    chars_per_line = max(
        1.0,
        usable_width / average_char_width,
    )

    lines = max(
        1.0,
        usable_height / line_height,
    )

    return chars_per_line * lines


def evaluate_pptx_layout(
    pptx_path: Path,
    expected_slide_count: int = 6,
) -> dict[str, float]:
    if not pptx_path.exists():
        return {
            "m_pptx_valid": 0.0,
            "m_pptx_slide_count": 0.0,
            "m_pptx_no_text_overlap": 0.0,
            "m_pptx_no_overflow_proxy": 0.0,
            "m_pptx_in_bounds": 0.0,
            "n_text_overlaps": 999.0,
            "n_overflow_proxy": 999.0,
            "n_out_of_bounds": 999.0,
            "n_slides": 0.0,
        }

    try:
        prs = Presentation(pptx_path)
    except Exception:
        return {
            "m_pptx_valid": 0.0,
            "m_pptx_slide_count": 0.0,
            "m_pptx_no_text_overlap": 0.0,
            "m_pptx_no_overflow_proxy": 0.0,
            "m_pptx_in_bounds": 0.0,
            "n_text_overlaps": 999.0,
            "n_overflow_proxy": 999.0,
            "n_out_of_bounds": 999.0,
            "n_slides": 0.0,
        }

    slide_width = emu_to_inches(
        prs.slide_width
    )
    slide_height = emu_to_inches(
        prs.slide_height
    )

    overlaps = 0
    overflow_proxy = 0
    out_of_bounds = 0

    for slide in prs.slides:
        text_shapes = []

        for shape in slide.shapes:
            text = shape_text(shape)

            if not text:
                continue

            text_shapes.append(shape)

            left, top, right, bottom = (
                text_shape_rect(shape)
            )

            if (
                left < -0.01
                or top < -0.01
                or right > slide_width + 0.01
                or bottom > slide_height + 0.01
            ):
                out_of_bounds += 1

            capacity = estimate_text_capacity(
                shape
            )

            # 15% запас из-за грубости оценки.
            if len(text) > capacity * 1.15:
                overflow_proxy += 1

        for index, shape_a in enumerate(
            text_shapes
        ):
            rect_a = text_shape_rect(shape_a)

            for shape_b in text_shapes[
                index + 1:
            ]:
                rect_b = text_shape_rect(shape_b)

                if (
                    intersection_ratio(
                        rect_a,
                        rect_b,
                    )
                    > 0.04
                ):
                    overlaps += 1

    slide_count = len(prs.slides)

    return {
        "m_pptx_valid": 1.0,
        "m_pptx_slide_count": max(
            0.0,
            1.0
            - abs(
                slide_count
                - expected_slide_count
            )
            / expected_slide_count,
        ),
        "m_pptx_no_text_overlap": max(
            0.0,
            1.0 - overlaps / 5.0,
        ),
        "m_pptx_no_overflow_proxy": max(
            0.0,
            1.0 - overflow_proxy / 5.0,
        ),
        "m_pptx_in_bounds": max(
            0.0,
            1.0 - out_of_bounds / 5.0,
        ),
        "n_text_overlaps": float(overlaps),
        "n_overflow_proxy": float(
            overflow_proxy
        ),
        "n_out_of_bounds": float(
            out_of_bounds
        ),
        "n_slides": float(slide_count),
    }


# ------------------------------------------------------------
# 9. Итоговый quality score
# ------------------------------------------------------------

QUALITY_WEIGHTS = {
    "m_product_precision": 0.08,
    "m_product_core_recall": 0.12,
    "m_product_forbidden_clean": 0.05,
    "m_product_mrr": 0.05,

    "m_news_precision": 0.07,
    "m_news_recall": 0.05,

    "m_no_fallback": 0.05,
    "m_generation_not_blocked": 0.04,
    "m_section_completeness": 0.06,
    "m_generated_product_grounding": 0.08,
    "m_generated_core_coverage": 0.08,
    "m_generated_forbidden_clean": 0.04,
    "m_generated_news_grounding": 0.05,
    "m_risk_keyword_coverage": 0.06,
    "m_plan_conciseness": 0.04,

    "m_pptx_valid": 0.03,
    "m_pptx_slide_count": 0.02,
    "m_pptx_no_text_overlap": 0.04,
    "m_pptx_no_overflow_proxy": 0.02,
    "m_pptx_in_bounds": 0.02,
}


def calculate_quality_score(
    metrics: dict[str, Any],
) -> float:
    weighted_sum = 0.0
    weight_sum = 0.0

    for metric_name, weight in (
        QUALITY_WEIGHTS.items()
    ):
        value = metrics.get(
            metric_name,
            0.0,
        )

        try:
            value = float(value)
        except (TypeError, ValueError):
            value = 0.0

        if not math.isfinite(value):
            value = 0.0

        value = min(
            1.0,
            max(0.0, value),
        )

        weighted_sum += value * weight
        weight_sum += weight

    if weight_sum == 0:
        return 0.0

    return round(
        100.0 * weighted_sum / weight_sum,
        2,
    )


# ------------------------------------------------------------
# 10. Запуск одной версии
# ------------------------------------------------------------

def json_safe_context(
    context: dict[str, Any],
) -> dict[str, Any]:
    return {
        "client_name": context["client_name"],
        "profile": context["profile"],
        "notes": context["notes"],
        "products": context["products"],
        "news": context["news"],
    }


def run_single_eval_case(
    version: AgentVersion,
    client_name: str,
    reuse_cache: bool = True,
) -> dict[str, Any]:
    case_dir = (
        EVAL_RUNS_DIR
        / version.name
        / safe_filename(client_name)
    )

    case_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = case_dir / "metrics.json"

    if (
        reuse_cache
        and metrics_path.exists()
    ):
        return json.loads(
            metrics_path.read_text(
                encoding="utf-8"
            )
        )

    started = time.perf_counter()

    row: dict[str, Any] = {
        "version": version.name,
        "client": client_name,
        "split": GOLD_CLIENT_SPECS[
            client_name
        ]["split"],
        "hard_failure": 0,
        "error": "",
    }

    try:
        retrieval_started = (
            time.perf_counter()
        )

        context = build_eval_context(
            version,
            client_name,
        )

        retrieval_latency = (
            time.perf_counter()
            - retrieval_started
        )

        plan, diagnostics = generate_eval_plan(
            context,
            version,
        )

        pptx_builder = (
            version.pptx_builder
            or create_pptx
        )

        pptx_path = (
            case_dir
            / f"{safe_filename(client_name)}.pptx"
        )

        pptx_started = time.perf_counter()

        pptx_builder(
            plan=plan,
            output_path=pptx_path,
        )

        pptx_latency = (
            time.perf_counter()
            - pptx_started
        )

        gold = GOLD_CLIENT_SPECS[
            client_name
        ]

        metrics = {
            **product_retrieval_metrics(
                context,
                gold,
            ),
            **news_retrieval_metrics(
                context,
                client_name,
            ),
            **generation_metrics(
                context,
                plan,
                diagnostics,
                gold,
            ),
            **evaluate_pptx_layout(
                pptx_path
            ),
        }

        row.update(metrics)

        row.update({
            "generation_mode": (
                diagnostics.mode
            ),
            "latency_retrieval_seconds": (
                retrieval_latency
            ),
            "latency_pptx_seconds": (
                pptx_latency
            ),
            "latency_total_seconds": (
                time.perf_counter()
                - started
            ),
            "pptx_path": str(pptx_path),
            "quality_score": 0.0,
        })

        row["quality_score"] = (
            calculate_quality_score(row)
        )

        (case_dir / "context.json").write_text(
            json.dumps(
                json_safe_context(context),
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        (case_dir / "plan.json").write_text(
            json.dumps(
                plan.model_dump(),
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

        (case_dir / "raw_response.txt").write_text(
            diagnostics.raw_response,
            encoding="utf-8",
        )

        (
            case_dir
            / "version_config.json"
        ).write_text(
            json.dumps(
                version.serializable_dict(),
                ensure_ascii=False,
                indent=2,
            ),
            encoding="utf-8",
        )

    except Exception as error:
        row.update({
            "hard_failure": 1,
            "error": (
                f"{type(error).__name__}: {error}"
            ),
            "quality_score": 0.0,
            "latency_total_seconds": (
                time.perf_counter()
                - started
            ),
        })

    metrics_path.write_text(
        json.dumps(
            row,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    return row


def evaluate_agent_version(
    version: AgentVersion,
    clients: list[str] | None = None,
    rebuild_index: bool = False,
    reuse_cache: bool = True,
) -> pd.DataFrame:
    ensure_index_ready(
        rebuild_index=rebuild_index
    )

    if clients is None:
        clients = list(
            GOLD_CLIENT_SPECS.keys()
        )

    print(
        f"\nОценка версии: {version.name}"
    )
    print(version.description)

    rows = []

    for index, client_name in enumerate(
        clients,
        start=1,
    ):
        print(
            f"[{index}/{len(clients)}] "
            f"{client_name}"
        )

        row = run_single_eval_case(
            version=version,
            client_name=client_name,
            reuse_cache=reuse_cache,
        )

        print(
            "  score:",
            row.get("quality_score"),
            "| mode:",
            row.get("generation_mode"),
            "| error:",
            row.get("error", ""),
        )

        rows.append(row)

    result = pd.DataFrame(rows)

    version_dir = (
        EVAL_RUNS_DIR / version.name
    )

    result.to_csv(
        version_dir / "summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    result.to_json(
        version_dir / "summary.json",
        orient="records",
        force_ascii=False,
        indent=2,
    )

    return result


# ------------------------------------------------------------
# 11. Быстрый retrieval-only прогон
# ------------------------------------------------------------

def evaluate_retrieval_only(
    version: AgentVersion,
    clients: list[str] | None = None,
) -> pd.DataFrame:
    ensure_index_ready(
        rebuild_index=False
    )

    if clients is None:
        clients = list(
            GOLD_CLIENT_SPECS.keys()
        )

    rows = []

    for client_name in clients:
        started = time.perf_counter()

        context = build_eval_context(
            version,
            client_name,
        )

        gold = GOLD_CLIENT_SPECS[
            client_name
        ]

        row = {
            "version": version.name,
            "client": client_name,
            "split": gold["split"],
            **product_retrieval_metrics(
                context,
                gold,
            ),
            **news_retrieval_metrics(
                context,
                client_name,
            ),
            "latency_seconds": (
                time.perf_counter()
                - started
            ),
        }

        rows.append(row)

    return pd.DataFrame(rows)


# ------------------------------------------------------------
# 12. Сравнение двух версий
# ------------------------------------------------------------

def bootstrap_mean_ci(
    values: np.ndarray,
    n_bootstrap: int = 5000,
    confidence: float = 0.95,
    seed: int = 42,
) -> tuple[float, float]:
    values = np.asarray(
        values,
        dtype=float,
    )

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return (float("nan"), float("nan"))

    rng = np.random.default_rng(seed)

    means = []

    for _ in range(n_bootstrap):
        sample = rng.choice(
            values,
            size=len(values),
            replace=True,
        )

        means.append(
            float(np.mean(sample))
        )

    alpha = 1.0 - confidence

    return (
        float(
            np.quantile(
                means,
                alpha / 2,
            )
        ),
        float(
            np.quantile(
                means,
                1 - alpha / 2,
            )
        ),
    )


def compare_agent_versions(
    baseline_df: pd.DataFrame,
    candidate_df: pd.DataFrame,
    baseline_name: str,
    candidate_name: str,
) -> dict[str, pd.DataFrame]:
    baseline = baseline_df.copy()
    candidate = candidate_df.copy()

    metric_columns = sorted(
        set(
            column
            for column in baseline.columns
            if column.startswith("m_")
        )
        & set(
            column
            for column in candidate.columns
            if column.startswith("m_")
        )
    )

    merged = baseline.merge(
        candidate,
        on=["client", "split"],
        suffixes=(
            f"_{baseline_name}",
            f"_{candidate_name}",
        ),
    )

    merged["quality_delta"] = (
        merged[
            f"quality_score_{candidate_name}"
        ]
        - merged[
            f"quality_score_{baseline_name}"
        ]
    )

    merged["result"] = np.select(
        [
            merged["quality_delta"] > 0.5,
            merged["quality_delta"] < -0.5,
        ],
        [
            "candidate_win",
            "candidate_loss",
        ],
        default="tie",
    )

    metric_delta_rows = []

    for metric in metric_columns:
        baseline_column = (
            f"{metric}_{baseline_name}"
        )
        candidate_column = (
            f"{metric}_{candidate_name}"
        )

        delta = (
            merged[candidate_column]
            - merged[baseline_column]
        )

        metric_delta_rows.append({
            "metric": metric,
            "baseline_mean": float(
                merged[
                    baseline_column
                ].mean()
            ),
            "candidate_mean": float(
                merged[
                    candidate_column
                ].mean()
            ),
            "delta": float(delta.mean()),
        })

    metric_deltas = pd.DataFrame(
        metric_delta_rows
    ).sort_values(
        "delta",
        ascending=False,
    )

    quality_ci = bootstrap_mean_ci(
        merged["quality_delta"].to_numpy()
    )

    summary = pd.DataFrame([
        {
            "baseline": baseline_name,
            "candidate": candidate_name,
            "baseline_mean_score": float(
                merged[
                    f"quality_score_{baseline_name}"
                ].mean()
            ),
            "candidate_mean_score": float(
                merged[
                    f"quality_score_{candidate_name}"
                ].mean()
            ),
            "mean_delta": float(
                merged["quality_delta"].mean()
            ),
            "delta_ci_low": quality_ci[0],
            "delta_ci_high": quality_ci[1],
            "candidate_wins": int(
                (
                    merged["result"]
                    == "candidate_win"
                ).sum()
            ),
            "candidate_losses": int(
                (
                    merged["result"]
                    == "candidate_loss"
                ).sum()
            ),
            "ties": int(
                (
                    merged["result"]
                    == "tie"
                ).sum()
            ),
            "baseline_hard_failures": int(
                merged[
                    f"hard_failure_{baseline_name}"
                ].sum()
            ),
            "candidate_hard_failures": int(
                merged[
                    f"hard_failure_{candidate_name}"
                ].sum()
            ),
        }
    ])

    timestamp = time.strftime(
        "%Y%m%d_%H%M%S"
    )

    comparison_dir = (
        EVAL_COMPARISONS_DIR
        / (
            f"{baseline_name}_vs_"
            f"{candidate_name}_{timestamp}"
        )
    )

    comparison_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    summary.to_csv(
        comparison_dir / "summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    merged.to_csv(
        comparison_dir / "paired_cases.csv",
        index=False,
        encoding="utf-8-sig",
    )

    metric_deltas.to_csv(
        comparison_dir / "metric_deltas.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return {
        "summary": summary,
        "paired_cases": merged,
        "metric_deltas": metric_deltas,
    }


# ------------------------------------------------------------
# 13. Human review template
# ------------------------------------------------------------

def create_human_review_template(
    results: pd.DataFrame,
    output_path: Path | None = None,
) -> Path:
    if output_path is None:
        output_path = (
            EVAL_DIR
            / "human_review_template.csv"
        )

    template = results[
        [
            "version",
            "client",
            "pptx_path",
        ]
    ].copy()

    template["factuality_1_5"] = ""
    template["client_specificity_1_5"] = ""
    template["product_fit_1_5"] = ""
    template["news_relevance_1_5"] = ""
    template["readability_1_5"] = ""
    template["comments"] = ""

    template.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    return output_path


Gold test set: D:\sber\evaluation\gold_test_set.json
